In [30]:
print("Spark session is working")

Spark session is working


In [31]:
from pyspark.sql import functions as F
from pyspark.sql.types import *

In [6]:
from pyspark.sql import functions as F

BUCKET = "mahmoud-sic-ecommerce-2026"

RAW_PATH = f"s3://{BUCKET}/raw/ecommerce"

PROCESSED_PATH = f"s3://{BUCKET}/processed/ecommerce"

TABLE_FILES = {
    "customers": "customers.csv",
    "categories": "categories.csv",
    "products": "products.csv",
    "departments": "departments.csv",
    "employees": "employees.csv",
    "suppliers": "suppliers.csv",
    "orders": "orders.csv",
    "order_details": "order_details.csv",
    "payments": "payments.csv",
    "product_suppliers": "product_suppliers.csv",
    "shippers": "shippers.csv",
    "shipments": "shipments.csv"
}

source_dfs = {}

for table_name, file_name in TABLE_FILES.items():
    path = f"{RAW_PATH}/{file_name}"

    df = (
        spark.read
        .option("header", True)
        .option("inferSchema", True)
        .option("multiLine", True)
        .csv(path)
    )

    source_dfs[table_name] = df

    print(
        f"{table_name}: "
        f"{df.count()} rows, "
        f"{len(df.columns)} columns"
    )


customers: 10000 rows, 7 columns
categories: 20 rows, 2 columns
products: 1000 rows, 7 columns
departments: 10 rows, 2 columns
employees: 200 rows, 7 columns
suppliers: 100 rows, 3 columns
orders: 50000 rows, 4 columns
order_details: 100000 rows, 6 columns
payments: 45000 rows, 5 columns
product_suppliers: 2027 rows, 2 columns
shippers: 10 rows, 2 columns
shipments: 40000 rows, 5 columns


In [7]:
id_checks = {
    "customers": "CustomerID",
    "products": "ProductID",
    "orders": "OrderID",
    "order_details": "OrderDetailID",
    "payments": "PaymentID",
    "shipments": "ShipmentID"
}

for table_name, id_column in id_checks.items():
    df = source_dfs[table_name]

    result = df.select(
        F.min(F.col(id_column)).alias("min_id"),
        F.max(F.col(id_column)).alias("max_id"),
        F.countDistinct(F.col(id_column)).alias("distinct_ids")
    )

    print(f"\n{table_name}.{id_column}")
    result.show()



customers.CustomerID
+------+------+------------+
|min_id|max_id|distinct_ids|
+------+------+------------+
|     1| 10000|       10000|
+------+------+------------+


products.ProductID
+------+------+------------+
|min_id|max_id|distinct_ids|
+------+------+------------+
|     1|  1000|        1000|
+------+------+------------+


orders.OrderID
+------+------+------------+
|min_id|max_id|distinct_ids|
+------+------+------------+
|     1| 50000|       50000|
+------+------+------------+


order_details.OrderDetailID
+------+------+------------+
|min_id|max_id|distinct_ids|
+------+------+------------+
|     1|100000|      100000|
+------+------+------------+


payments.PaymentID
+------+------+------------+
|min_id|max_id|distinct_ids|
+------+------+------------+
|     1| 45000|       45000|
+------+------+------------+


shipments.ShipmentID
+------+------+------------+
|min_id|max_id|distinct_ids|
+------+------+------------+
|     1| 40000|       40000|
+------+------+----------

In [3]:
# ============================================================
# STEP 2 - DATA PROFILING
# Schema, Samples, Nulls, and Duplicate Keys
# ============================================================

PRIMARY_KEYS = {
    "customers": "CustomerID",
    "categories": "CategoryID",
    "products": "ProductID",
    "departments": "DepartmentID",
    "employees": "EmployeeID",
    "suppliers": "SupplierID",
    "orders": "OrderID",
    "order_details": "OrderDetailID",
    "payments": "PaymentID",
    "product_suppliers": None,
    "shippers": "ShipperID",
    "shipments": "ShipmentID"
}

for table_name, df in source_dfs.items():

    print("\n" + "=" * 100)
    print(f"TABLE: {table_name}")
    print("=" * 100)

    print("\nSchema:")
    df.printSchema()

    print("\nSample records:")
    df.show(10, truncate=False)

    print("\nNull counts:")
    null_count_df = df.select([
        F.sum(
            F.when(F.col(column_name).isNull(), 1).otherwise(0)
        ).alias(column_name)
        for column_name in df.columns
    ])

    null_count_df.show(truncate=False)

    primary_key = PRIMARY_KEYS.get(table_name)

    if primary_key is not None:
        duplicate_count = (
            df.groupBy(primary_key)
              .count()
              .filter(F.col("count") > 1)
              .count()
        )

        print(
            f"Duplicate {primary_key} values: "
            f"{duplicate_count}"
        )
    else:
        duplicate_count = (
            df.groupBy("ProductID", "SupplierID")
              .count()
              .filter(F.col("count") > 1)
              .count()
        )

        print(
            "Duplicate ProductID + SupplierID pairs: "
            f"{duplicate_count}"
        )



TABLE: customers

Schema:
root
 |-- CustomerID: integer (nullable = true)
 |-- FirstName: string (nullable = true)
 |-- LastName: string (nullable = true)
 |-- Email: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- RegistrationDate: timestamp (nullable = true)


Sample records:
+----------+---------+--------+-----------------------------+----------+--------------+-------------------+
|CustomerID|FirstName|LastName|Email                        |City      |Country       |RegistrationDate   |
+----------+---------+--------+-----------------------------+----------+--------------+-------------------+
|1         |Danielle |Johnson |danielle.johnson1@example.com|Giza      |Egypt         |2023-01-31 00:00:00|
|2         |Joshua   |Walker  |joshua.walker2@example.com   |Dubai     |Jordan        |2021-07-25 00:00:00|
|3         |Jill     |Rhodes  |jill.rhodes3@example.com     |Giza      |United Kingdom|2020-12-22 00:00:00|
|4         |Pat

In [4]:
# ============================================================
# STEP 3 - INVALID VALUES AND FOREIGN KEY CHECKS
# ============================================================

def print_count(label, count_value):
    print(f"{label}: {count_value}")


# ------------------------------------------------------------
# 1. Negative product prices and costs
# ------------------------------------------------------------

products_df = source_dfs["products"]

print_count(
    "Products with negative price",
    products_df.filter(F.col("Price") < 0).count()
)

print_count(
    "Products with negative cost",
    products_df.filter(F.col("Cost") < 0).count()
)


# ------------------------------------------------------------
# 2. Negative quantities and prices in order details
# ------------------------------------------------------------

order_details_df = source_dfs["order_details"]

print_count(
    "Order details with negative quantity",
    order_details_df.filter(F.col("Quantity") < 0).count()
)

print_count(
    "Order details with negative unit price",
    order_details_df.filter(F.col("UnitPrice") < 0).count()
)

print_count(
    "Order details with negative discount",
    order_details_df.filter(F.col("Discount") < 0).count()
)


# ------------------------------------------------------------
# 3. Negative payment amounts
# ------------------------------------------------------------

payments_df = source_dfs["payments"]

print_count(
    "Payments with negative amount",
    payments_df.filter(F.col("Amount") < 0).count()
)


# ------------------------------------------------------------
# 4. Invalid customer emails
# ------------------------------------------------------------

customers_df = source_dfs["customers"]

email_pattern = r"^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$"

invalid_emails = (
    customers_df
    .filter(
        F.col("Email").isNull()
        | (~F.lower(F.trim(F.col("Email"))).rlike(email_pattern))
    )
)

print_count(
    "Invalid or null customer emails",
    invalid_emails.count()
)


# ------------------------------------------------------------
# 5. Invalid order statuses
# ------------------------------------------------------------

valid_order_statuses = [
    "Pending",
    "Processing",
    "Shipped",
    "Delivered",
    "Cancelled",
    "Confirmed"
]

orders_df = source_dfs["orders"]

invalid_statuses = (
    orders_df
    .filter(
        F.col("Status").isNull()
        | (~F.initcap(F.trim(F.col("Status"))).isin(valid_order_statuses))
    )
)

print_count(
    "Invalid or null order statuses",
    invalid_statuses.count()
)

print("Existing order statuses:")
orders_df.groupBy("Status").count().orderBy("Status").show(
    truncate=False
)


# ------------------------------------------------------------
# 6. Invalid shipment dates
# ------------------------------------------------------------

shipments_df = source_dfs["shipments"]

invalid_shipments_dates = (
    shipments_df
    .filter(
        F.col("ShipDate").isNull()
        | F.col("DeliveryDate").isNull()
        | (
            F.to_date(F.col("DeliveryDate"))
            < F.to_date(F.col("ShipDate"))
        )
    )
)

print_count(
    "Shipments with invalid dates",
    invalid_shipments_dates.count()
)


# ------------------------------------------------------------
# 7. Foreign key checks
# ------------------------------------------------------------

customers_keys = customers_df.select("CustomerID").distinct()
products_keys = products_df.select("ProductID").distinct()
orders_keys = orders_df.select("OrderID").distinct()
shippers_keys = source_dfs["shippers"].select("ShipperID").distinct()


# Orders referencing nonexistent customers
orphan_orders = (
    orders_df
    .select("OrderID", "CustomerID")
    .join(customers_keys, on="CustomerID", how="left_anti")
)

print_count(
    "Orders with nonexistent customers",
    orphan_orders.count()
)


# Order details referencing nonexistent orders
orphan_order_details_orders = (
    order_details_df
    .select("OrderDetailID", "OrderID")
    .join(orders_keys, on="OrderID", how="left_anti")
)

print_count(
    "Order details with nonexistent orders",
    orphan_order_details_orders.count()
)


# Order details referencing nonexistent products
orphan_order_details_products = (
    order_details_df
    .select("OrderDetailID", "ProductID")
    .join(products_keys, on="ProductID", how="left_anti")
)

print_count(
    "Order details with nonexistent products",
    orphan_order_details_products.count()
)


# Payments referencing nonexistent orders
orphan_payments = (
    payments_df
    .select("PaymentID", "OrderID")
    .join(orders_keys, on="OrderID", how="left_anti")
)

print_count(
    "Payments with nonexistent orders",
    orphan_payments.count()
)


# Shipments referencing nonexistent orders
orphan_shipments_orders = (
    shipments_df
    .select("ShipmentID", "OrderID")
    .join(orders_keys, on="OrderID", how="left_anti")
)

print_count(
    "Shipments with nonexistent orders",
    orphan_shipments_orders.count()
)


# Shipments referencing nonexistent shippers
orphan_shipments_shippers = (
    shipments_df
    .select("ShipmentID", "ShipperID")
    .join(shippers_keys, on="ShipperID", how="left_anti")
)

print_count(
    "Shipments with nonexistent shippers",
    orphan_shipments_shippers.count()
)


Products with negative price: 0
Products with negative cost: 0
Order details with negative quantity: 0
Order details with negative unit price: 0
Order details with negative discount: 0
Payments with negative amount: 0
Invalid or null customer emails: 0
Invalid or null order statuses: 0
Existing order statuses:
+----------+-----+
|Status    |count|
+----------+-----+
|Cancelled |9998 |
|Delivered |9975 |
|Pending   |10044|
|Processing|9968 |
|Shipped   |10015|
+----------+-----+

Shipments with invalid dates: 0
Orders with nonexistent customers: 0
Order details with nonexistent orders: 0
Order details with nonexistent products: 0
Payments with nonexistent orders: 0
Shipments with nonexistent orders: 0
Shipments with nonexistent shippers: 0


In [8]:
# ============================================================
# STEP 4 - CLEAN AND STANDARDIZE SOURCE TABLES
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.types import StringType


def trim_string_columns(df):
    """
    Remove leading/trailing spaces from all string columns.
    """
    for field in df.schema.fields:
        if isinstance(field.dataType, StringType):
            df = df.withColumn(
                field.name,
                F.trim(F.col(field.name))
            )
    return df


def standardize_table(df, primary_key, composite_key=None):
    """
    Standardize column names, trim strings,
    remove null primary keys, and remove duplicates.
    """

    # Convert column names to lowercase
    for column_name in df.columns:
        df = df.withColumnRenamed(
            column_name,
            column_name.strip().lower()
        )

    # Remove unnecessary spaces from string columns
    df = trim_string_columns(df)

    # Remove rows with null primary key
    if primary_key is not None:
        df = df.filter(F.col(primary_key).isNotNull())

    # Remove duplicate records based on business key
    if composite_key is not None:
        df = df.dropDuplicates(composite_key)
    elif primary_key is not None:
        df = df.dropDuplicates([primary_key])
    else:
        df = df.dropDuplicates()

    return df


clean_dfs = {}

# ------------------------------------------------------------
# Customers
# ------------------------------------------------------------

customers_clean = standardize_table(
    source_dfs["customers"],
    primary_key="customerid"
)

customers_clean = (
    customers_clean
    .withColumn("customerid", F.col("customerid").cast("long"))
    .withColumn("firstname", F.trim(F.col("firstname")))
    .withColumn("lastname", F.trim(F.col("lastname")))
    .withColumn("email", F.lower(F.trim(F.col("email"))))
    .withColumn("city", F.trim(F.col("city")))
    .withColumn("country", F.trim(F.col("country")))
    .withColumn(
        "registrationdate",
        F.to_date(F.col("registrationdate"))
    )
)

clean_dfs["customers"] = customers_clean


# ------------------------------------------------------------
# Categories
# ------------------------------------------------------------

categories_clean = standardize_table(
    source_dfs["categories"],
    primary_key="categoryid"
)

categories_clean = (
    categories_clean
    .withColumn("categoryid", F.col("categoryid").cast("long"))
    .withColumn(
        "categoryname",
        F.trim(F.col("categoryname"))
    )
)

clean_dfs["categories"] = categories_clean


# ------------------------------------------------------------
# Departments
# ------------------------------------------------------------

departments_clean = standardize_table(
    source_dfs["departments"],
    primary_key="departmentid"
)

departments_clean = (
    departments_clean
    .withColumn("departmentid", F.col("departmentid").cast("long"))
    .withColumn(
        "departmentname",
        F.trim(F.col("departmentname"))
    )
)

clean_dfs["departments"] = departments_clean


# ------------------------------------------------------------
# Products
# ------------------------------------------------------------

products_clean = standardize_table(
    source_dfs["products"],
    primary_key="productid"
)

products_clean = (
    products_clean
    .withColumn("productid", F.col("productid").cast("long"))
    .withColumn("categoryid", F.col("categoryid").cast("long"))
    .withColumn("productname", F.trim(F.col("productname")))
    .withColumn("brand", F.trim(F.col("brand")))
    .withColumn(
        "price",
        F.col("price").cast("decimal(12,2)")
    )
    .withColumn(
        "cost",
        F.col("cost").cast("decimal(12,2)")
    )
    .withColumn("stock", F.col("stock").cast("int"))
)

clean_dfs["products"] = products_clean


# ------------------------------------------------------------
# Employees
# ------------------------------------------------------------

employees_clean = standardize_table(
    source_dfs["employees"],
    primary_key="employeeid"
)

employees_clean = (
    employees_clean
    .withColumn("employeeid", F.col("employeeid").cast("long"))
    .withColumn("managerid", F.col("managerid").cast("long"))
    .withColumn("departmentid", F.col("departmentid").cast("long"))
    .withColumn("firstname", F.trim(F.col("firstname")))
    .withColumn("lastname", F.trim(F.col("lastname")))
    .withColumn(
        "salary",
        F.col("salary").cast("decimal(12,2)")
    )
    .withColumn(
        "hiredate",
        F.to_date(F.col("hiredate"))
    )
)

clean_dfs["employees"] = employees_clean


# ------------------------------------------------------------
# Suppliers
# ------------------------------------------------------------

suppliers_clean = standardize_table(
    source_dfs["suppliers"],
    primary_key="supplierid"
)

suppliers_clean = (
    suppliers_clean
    .withColumn("supplierid", F.col("supplierid").cast("long"))
    .withColumn(
        "suppliername",
        F.trim(F.col("suppliername"))
    )
    .withColumn("country", F.trim(F.col("country")))
)

clean_dfs["suppliers"] = suppliers_clean


# ------------------------------------------------------------
# Orders
# ------------------------------------------------------------

orders_clean = standardize_table(
    source_dfs["orders"],
    primary_key="orderid"
)

orders_clean = (
    orders_clean
    .withColumn("orderid", F.col("orderid").cast("long"))
    .withColumn("customerid", F.col("customerid").cast("long"))
    .withColumn(
        "orderdate",
        F.to_date(F.col("orderdate"))
    )
    .withColumn(
        "status",
        F.upper(F.trim(F.col("status")))
    )
)

clean_dfs["orders"] = orders_clean


# ------------------------------------------------------------
# Order Details
# ------------------------------------------------------------

order_details_clean = standardize_table(
    source_dfs["order_details"],
    primary_key="orderdetailid"
)

order_details_clean = (
    order_details_clean
    .withColumn(
        "orderdetailid",
        F.col("orderdetailid").cast("long")
    )
    .withColumn("orderid", F.col("orderid").cast("long"))
    .withColumn("productid", F.col("productid").cast("long"))
    .withColumn("quantity", F.col("quantity").cast("int"))
    .withColumn(
        "unitprice",
        F.col("unitprice").cast("decimal(12,2)")
    )
    .withColumn(
        "discount",
        F.col("discount").cast("decimal(5,2)")
    )
)

clean_dfs["order_details"] = order_details_clean


# ------------------------------------------------------------
# Payments
# ------------------------------------------------------------

payments_clean = standardize_table(
    source_dfs["payments"],
    primary_key="paymentid"
)

payments_clean = (
    payments_clean
    .withColumn("paymentid", F.col("paymentid").cast("long"))
    .withColumn("orderid", F.col("orderid").cast("long"))
    .withColumn(
        "paymentmethod",
        F.upper(F.trim(F.col("paymentmethod")))
    )
    .withColumn(
        "paymentdate",
        F.to_date(F.col("paymentdate"))
    )
    .withColumn(
        "amount",
        F.col("amount").cast("decimal(14,2)")
    )
)

clean_dfs["payments"] = payments_clean


# ------------------------------------------------------------
# Product Suppliers
# ------------------------------------------------------------

product_suppliers_clean = standardize_table(
    source_dfs["product_suppliers"],
    primary_key=None,
    composite_key=["productid", "supplierid"]
)

product_suppliers_clean = (
    product_suppliers_clean
    .withColumn("productid", F.col("productid").cast("long"))
    .withColumn("supplierid", F.col("supplierid").cast("long"))
)

clean_dfs["product_suppliers"] = product_suppliers_clean


# ------------------------------------------------------------
# Shippers
# ------------------------------------------------------------

shippers_clean = standardize_table(
    source_dfs["shippers"],
    primary_key="shipperid"
)

shippers_clean = (
    shippers_clean
    .withColumn("shipperid", F.col("shipperid").cast("long"))
    .withColumn(
        "companyname",
        F.trim(F.col("companyname"))
    )
)

clean_dfs["shippers"] = shippers_clean


# ------------------------------------------------------------
# Shipments
# ------------------------------------------------------------

shipments_clean = standardize_table(
    source_dfs["shipments"],
    primary_key="shipmentid"
)

shipments_clean = (
    shipments_clean
    .withColumn("shipmentid", F.col("shipmentid").cast("long"))
    .withColumn("orderid", F.col("orderid").cast("long"))
    .withColumn("shipperid", F.col("shipperid").cast("long"))
    .withColumn(
        "shipdate",
        F.to_date(F.col("shipdate"))
    )
    .withColumn(
        "deliverydate",
        F.to_date(F.col("deliverydate"))
    )
)

clean_dfs["shipments"] = shipments_clean


print("All source tables were cleaned successfully.")


All source tables were cleaned successfully.


In [9]:
# ============================================================
# STEP 5 - BUILD BASIC DIMENSIONS
# ============================================================

from pyspark.sql import Window
from pyspark.sql import functions as F

dim_window = Window.orderBy("natural_key")


# ------------------------------------------------------------
# dim_customer
# Grain: one row per customer
# ------------------------------------------------------------

dim_customer = (
    clean_dfs["customers"]
    .select(
        F.col("customerid").alias("natural_key"),
        F.col("customerid").alias("customer_id"),
        F.col("firstname").alias("first_name"),
        F.col("lastname").alias("last_name"),
        F.concat_ws(
            " ",
            F.col("firstname"),
            F.col("lastname")
        ).alias("full_name"),
        F.col("email"),
        F.col("city"),
        F.col("country"),
        F.col("registrationdate").alias("registration_date")
    )
    .dropDuplicates(["customer_id"])
    .withColumn(
        "customer_key",
        F.row_number().over(dim_window)
    )
    .select(
        "customer_key",
        "customer_id",
        "first_name",
        "last_name",
        "full_name",
        "email",
        "city",
        "country",
        "registration_date"
    )
)


# ------------------------------------------------------------
# dim_category
# Grain: one row per category
# ------------------------------------------------------------

dim_category = (
    clean_dfs["categories"]
    .select(
        F.col("categoryid").alias("natural_key"),
        F.col("categoryid").alias("category_id"),
        F.col("categoryname").alias("category_name")
    )
    .dropDuplicates(["category_id"])
    .withColumn(
        "category_key",
        F.row_number().over(dim_window)
    )
    .select(
        "category_key",
        "category_id",
        "category_name"
    )
)


# ------------------------------------------------------------
# dim_department
# Grain: one row per department
# ------------------------------------------------------------

dim_department = (
    clean_dfs["departments"]
    .select(
        F.col("departmentid").alias("natural_key"),
        F.col("departmentid").alias("department_id"),
        F.col("departmentname").alias("department_name")
    )
    .dropDuplicates(["department_id"])
    .withColumn(
        "department_key",
        F.row_number().over(dim_window)
    )
    .select(
        "department_key",
        "department_id",
        "department_name"
    )
)


# ------------------------------------------------------------
# dim_supplier
# Grain: one row per supplier
# ------------------------------------------------------------

dim_supplier = (
    clean_dfs["suppliers"]
    .select(
        F.col("supplierid").alias("natural_key"),
        F.col("supplierid").alias("supplier_id"),
        F.col("suppliername").alias("supplier_name"),
        F.col("country")
    )
    .dropDuplicates(["supplier_id"])
    .withColumn(
        "supplier_key",
        F.row_number().over(dim_window)
    )
    .select(
        "supplier_key",
        "supplier_id",
        "supplier_name",
        "country"
    )
)


# ------------------------------------------------------------
# dim_employee
# Grain: one row per employee
# ------------------------------------------------------------

dim_employee = (
    clean_dfs["employees"]
    .select(
        F.col("employeeid").alias("natural_key"),
        F.col("employeeid").alias("employee_id"),
        F.col("managerid").alias("manager_id"),
        F.col("departmentid").alias("department_id"),
        F.col("firstname").alias("first_name"),
        F.col("lastname").alias("last_name"),
        F.concat_ws(
            " ",
            F.col("firstname"),
            F.col("lastname")
        ).alias("full_name"),
        F.col("salary"),
        F.col("hiredate").alias("hire_date")
    )
    .dropDuplicates(["employee_id"])
    .withColumn(
        "employee_key",
        F.row_number().over(dim_window)
    )
    .select(
        "employee_key",
        "employee_id",
        "manager_id",
        "department_id",
        "first_name",
        "last_name",
        "full_name",
        "salary",
        "hire_date"
    )
)


# ------------------------------------------------------------
# dim_shipper
# Grain: one row per shipper
# ------------------------------------------------------------

dim_shipper = (
    clean_dfs["shippers"]
    .select(
        F.col("shipperid").alias("natural_key"),
        F.col("shipperid").alias("shipper_id"),
        F.col("companyname").alias("company_name")
    )
    .dropDuplicates(["shipper_id"])
    .withColumn(
        "shipper_key",
        F.row_number().over(dim_window)
    )
    .select(
        "shipper_key",
        "shipper_id",
        "company_name"
    )
)


print("Basic dimensions created successfully.")


Basic dimensions created successfully.


In [7]:
# ============================================================
# STEP 5B - VERIFY BASIC DIMENSIONS
# ============================================================

dimension_dfs = {
    "dim_customer": dim_customer,
    "dim_category": dim_category,
    "dim_department": dim_department,
    "dim_supplier": dim_supplier,
    "dim_employee": dim_employee,
    "dim_shipper": dim_shipper
}

for dimension_name, df in dimension_dfs.items():
    print(f"\n{dimension_name}: {df.count()} rows")
    df.show(3, truncate=False)



dim_customer: 10000 rows
+------------+-----------+----------+---------+----------------+-----------------------------+-----+--------------+-----------------+
|customer_key|customer_id|first_name|last_name|full_name       |email                        |city |country       |registration_date|
+------------+-----------+----------+---------+----------------+-----------------------------+-----+--------------+-----------------+
|1           |1          |Danielle  |Johnson  |Danielle Johnson|danielle.johnson1@example.com|Giza |Egypt         |2023-01-31       |
|2           |2          |Joshua    |Walker   |Joshua Walker   |joshua.walker2@example.com   |Dubai|Jordan        |2021-07-25       |
|3           |3          |Jill      |Rhodes   |Jill Rhodes     |jill.rhodes3@example.com     |Giza |United Kingdom|2020-12-22       |
+------------+-----------+----------+---------+----------------+-----------------------------+-----+--------------+-----------------+
only showing top 3 rows


dim_catego

In [10]:
# ============================================================
# STEP 6 - BUILD REMAINING DIMENSIONS
# ============================================================

from pyspark.sql import Window
from pyspark.sql import functions as F
from pyspark.sql.types import LongType


# ------------------------------------------------------------
# dim_product
# Grain: one row per product
# ------------------------------------------------------------

product_window = Window.orderBy("product_id")

dim_product_base = (
    clean_dfs["products"]
    .select(
        F.col("productid").alias("product_id"),
        F.col("categoryid").alias("category_id"),
        F.col("productname").alias("product_name"),
        F.col("brand"),
        F.col("price"),
        F.col("cost"),
        F.col("stock")
    )
    .dropDuplicates(["product_id"])
)

dim_product = (
    dim_product_base
    .join(
        dim_category.select(
            "category_key",
            "category_id"
        ),
        on="category_id",
        how="left"
    )
    .withColumn(
        "product_key",
        F.row_number().over(product_window)
    )
    .withColumn(
        "profit_per_unit",
        (
            F.col("price") - F.col("cost")
        ).cast("decimal(14,2)")
    )
    .withColumn(
        "profit_margin",
        F.when(
            F.col("price") > 0,
            (
                (F.col("price") - F.col("cost"))
                / F.col("price")
            ).cast("decimal(8,4)")
        ).otherwise(F.lit(None))
    )
    .withColumn(
        "department_key",
        F.lit(None).cast("long")
    )
    .select(
        "product_key",
        "product_id",
        "product_name",
        "brand",
        "category_key",
        "department_key",
        "price",
        "cost",
        "stock",
        "profit_per_unit",
        "profit_margin"
    )
)


# ------------------------------------------------------------
# dim_payment_method
# Grain: one row per payment method
# ------------------------------------------------------------

payment_method_window = Window.orderBy("payment_method")

dim_payment_method = (
    clean_dfs["payments"]
    .select(
        F.upper(
            F.trim(F.col("paymentmethod"))
        ).alias("payment_method")
    )
    .filter(F.col("payment_method").isNotNull())
    .dropDuplicates(["payment_method"])
    .withColumn(
        "payment_method_key",
        F.row_number().over(payment_method_window)
    )
    .select(
        "payment_method_key",
        "payment_method"
    )
)


# ------------------------------------------------------------
# dim_order_status
# Grain: one row per order status
# ------------------------------------------------------------

order_status_window = Window.orderBy("order_status")

dim_order_status = (
    clean_dfs["orders"]
    .select(
        F.upper(
            F.trim(F.col("status"))
        ).alias("order_status")
    )
    .filter(F.col("order_status").isNotNull())
    .dropDuplicates(["order_status"])
    .withColumn(
        "order_status_key",
        F.row_number().over(order_status_window)
    )
    .select(
        "order_status_key",
        "order_status"
    )
)


# ------------------------------------------------------------
# dim_date
# Grain: one row per calendar date
# ------------------------------------------------------------

order_dates = (
    clean_dfs["orders"]
    .select(F.col("orderdate").alias("full_date"))
)

payment_dates = (
    clean_dfs["payments"]
    .select(F.col("paymentdate").alias("full_date"))
)

shipment_dates = (
    clean_dfs["shipments"]
    .select(F.col("shipdate").alias("full_date"))
    .unionByName(
        clean_dfs["shipments"]
        .select(F.col("deliverydate").alias("full_date"))
    )
)

all_source_dates = (
    order_dates
    .unionByName(payment_dates)
    .unionByName(shipment_dates)
    .filter(F.col("full_date").isNotNull())
)

date_range = (
    all_source_dates
    .agg(
        F.min("full_date").alias("min_date"),
        F.max("full_date").alias("max_date")
    )
)

dim_date = (
    date_range
    .select(
        F.explode(
            F.sequence(
                F.col("min_date"),
                F.col("max_date"),
                F.expr("interval 1 day")
            )
        ).alias("full_date")
    )
    .withColumn(
        "date_key",
        F.date_format(
            F.col("full_date"),
            "yyyyMMdd"
        ).cast("int")
    )
    .withColumn(
        "day",
        F.dayofmonth(F.col("full_date"))
    )
    .withColumn(
        "month",
        F.month(F.col("full_date"))
    )
    .withColumn(
        "month_name",
        F.date_format(F.col("full_date"), "MMMM")
    )
    .withColumn(
        "quarter",
        F.quarter(F.col("full_date"))
    )
    .withColumn(
        "year",
        F.year(F.col("full_date"))
    )
    .withColumn(
        "week",
        F.weekofyear(F.col("full_date"))
    )
    .withColumn(
        "day_name",
        F.date_format(F.col("full_date"), "EEEE")
    )
    .withColumn(
        "is_weekend",
        F.dayofweek(F.col("full_date")).isin([1, 7])
    )
    .select(
        "date_key",
        "full_date",
        "day",
        "month",
        "month_name",
        "quarter",
        "year",
        "week",
        "day_name",
        "is_weekend"
    )
)

print("Remaining dimensions created successfully.")


Remaining dimensions created successfully.


In [9]:
# ============================================================
# STEP 6B - VERIFY REMAINING DIMENSIONS
# ============================================================

remaining_dimension_dfs = {
    "dim_product": dim_product,
    "dim_payment_method": dim_payment_method,
    "dim_order_status": dim_order_status,
    "dim_date": dim_date
}

for dimension_name, df in remaining_dimension_dfs.items():
    print(f"\n{dimension_name}: {df.count()} rows")
    df.show(5, truncate=False)



dim_product: 1000 rows
+-----------+----------+-----------------+------+------------+--------------+-------+-------+-----+---------------+-------------+
|product_key|product_id|product_name     |brand |category_key|department_key|price  |cost   |stock|profit_per_unit|profit_margin|
+-----------+----------+-----------------+------+------------+--------------+-------+-------+-----+---------------+-------------+
|1          |1         |Bose Printer 1   |Bose  |12          |null          |261.16 |140.65 |399  |120.51         |0.4614       |
|2          |2         |Lenovo Mouse 2   |Lenovo|3           |null          |1874.20|1460.92|155  |413.28         |0.2205       |
|3          |3         |Sony Headphones 3|Sony  |12          |null          |1272.54|846.77 |315  |425.77         |0.3346       |
|4          |4         |LG Smart Watch 4 |LG    |11          |null          |1180.67|637.81 |95   |542.86         |0.4598       |
|5          |5         |Bose Television 5|Bose  |16          |null

In [10]:
# ============================================================
# STEP 7 - BUILD FACT_ORDER_DETAIL
# Grain: one row per product line within an order
# ============================================================

orders_for_fact = (
    clean_dfs["orders"]
    .select(
        F.col("orderid").alias("order_id"),
        F.col("customerid").alias("customer_id"),
        F.col("orderdate").alias("order_date"),
        F.col("status").alias("order_status")
    )
)

details_for_fact = (
    clean_dfs["order_details"]
    .select(
        F.col("orderdetailid").alias("order_detail_id"),
        F.col("orderid").alias("order_id"),
        F.col("productid").alias("product_id"),
        F.col("quantity"),
        F.col("unitprice").alias("unit_price"),
        F.col("discount")
    )
)

product_for_fact = (
    dim_product
    .select(
        "product_key",
        "product_id",
        "cost"
    )
)

customer_for_fact = (
    dim_customer
    .select(
        "customer_key",
        "customer_id"
    )
)

date_for_fact = (
    dim_date
    .select(
        "date_key",
        F.col("full_date").alias("order_date")
    )
)

status_for_fact = (
    dim_order_status
    .select(
        "order_status_key",
        "order_status"
    )
)

fact_order_detail = (
    details_for_fact
    .join(
        orders_for_fact,
        on="order_id",
        how="left"
    )
    .join(
        product_for_fact,
        on="product_id",
        how="left"
    )
    .join(
        customer_for_fact,
        on="customer_id",
        how="left"
    )
    .join(
        date_for_fact,
        on="order_date",
        how="left"
    )
    .join(
        status_for_fact,
        on="order_status",
        how="left"
    )
    .withColumn(
        "gross_sales",
        (
            F.col("quantity") * F.col("unit_price")
        ).cast("decimal(16,2)")
    )
    .withColumn(
        "discount_amount",
        (
            F.col("quantity")
            * F.col("unit_price")
            * F.col("discount")
            / F.lit(100)
        ).cast("decimal(16,2)")
    )
    .withColumn(
        "net_sales",
        (
            F.col("gross_sales")
            - F.col("discount_amount")
        ).cast("decimal(16,2)")
    )
    .withColumn(
        "cost_amount",
        (
            F.col("quantity") * F.col("cost")
        ).cast("decimal(16,2)")
    )
    .withColumn(
        "profit",
        (
            F.col("net_sales")
            - F.col("cost_amount")
        ).cast("decimal(16,2)")
    )
    .select(
        "order_detail_id",
        "order_id",
        "product_key",
        "product_id",
        "customer_key",
        "customer_id",
        "date_key",
        "order_date",
        "order_status_key",
        "quantity",
        "unit_price",
        "discount",
        "gross_sales",
        "discount_amount",
        "net_sales",
        "cost_amount",
        "profit"
    )
)

print("fact_order_detail created successfully.")


fact_order_detail created successfully.


In [11]:
print(f"fact_order_detail rows: {fact_order_detail.count()}")
fact_order_detail.printSchema()
fact_order_detail.show(5, truncate=False)

print("Null foreign-key counts:")

fact_order_detail.select([
    F.sum(
        F.when(F.col(column_name).isNull(), 1).otherwise(0)
    ).alias(column_name)
    for column_name in [
        "product_key",
        "customer_key",
        "date_key",
        "order_status_key"
    ]
]).show()


fact_order_detail rows: 100000
root
 |-- order_detail_id: long (nullable = true)
 |-- order_id: long (nullable = true)
 |-- product_key: integer (nullable = true)
 |-- product_id: long (nullable = true)
 |-- customer_key: integer (nullable = true)
 |-- customer_id: long (nullable = true)
 |-- date_key: integer (nullable = true)
 |-- order_date: date (nullable = true)
 |-- order_status_key: integer (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: decimal(12,2) (nullable = true)
 |-- discount: decimal(5,2) (nullable = true)
 |-- gross_sales: decimal(16,2) (nullable = true)
 |-- discount_amount: decimal(16,2) (nullable = true)
 |-- net_sales: decimal(16,2) (nullable = true)
 |-- cost_amount: decimal(16,2) (nullable = true)
 |-- profit: decimal(16,2) (nullable = true)

+---------------+--------+-----------+----------+------------+-----------+--------+----------+----------------+--------+----------+--------+-----------+---------------+---------+-----------+-------

In [14]:
# ============================================================
# STEP 8 - CORRECT FACT_ORDER
# Grain: one row per order from the orders source table
# ============================================================

# All historical orders must be the driving table
orders_base = (
    clean_dfs["orders"]
    .select(
        F.col("orderid").alias("order_id"),
        F.col("customerid").alias("customer_id"),
        F.col("orderdate").alias("order_date"),
        F.col("status").alias("order_status")
    )
)

# Aggregate available order-detail measures
order_detail_metrics = (
    fact_order_detail
    .groupBy("order_id")
    .agg(
        F.countDistinct("order_detail_id").alias("line_count"),
        F.countDistinct("product_id").alias("product_count"),
        F.sum("quantity").alias("total_quantity"),
        F.sum("gross_sales").cast("decimal(16,2)").alias("gross_sales"),
        F.sum("discount_amount").cast("decimal(16,2)").alias("discount_amount"),
        F.sum("net_sales").cast("decimal(16,2)").alias("net_sales"),
        F.sum("cost_amount").cast("decimal(16,2)").alias("cost_amount"),
        F.sum("profit").cast("decimal(16,2)").alias("profit")
    )
)

# Dimension lookups
customer_lookup = dim_customer.select(
    "customer_key",
    "customer_id"
)

date_lookup = dim_date.select(
    "date_key",
    F.col("full_date").alias("order_date")
)

status_lookup = dim_order_status.select(
    "order_status_key",
    "order_status"
)

# Build one row for every order
fact_order = (
    orders_base
    .join(
        customer_lookup,
        on="customer_id",
        how="left"
    )
    .join(
        date_lookup,
        on="order_date",
        how="left"
    )
    .join(
        status_lookup,
        on="order_status",
        how="left"
    )
    .join(
        order_detail_metrics,
        on="order_id",
        how="left"
    )
    .select(
        "order_id",
        "customer_key",
        "customer_id",
        "date_key",
        "order_date",
        "order_status_key",
        "order_status",
        F.coalesce(F.col("line_count"), F.lit(0)).alias("line_count"),
        F.coalesce(F.col("product_count"), F.lit(0)).alias("product_count"),
        F.coalesce(F.col("total_quantity"), F.lit(0)).alias("total_quantity"),
        F.coalesce(
            F.col("gross_sales"),
            F.lit(0).cast("decimal(16,2)")
        ).alias("gross_sales"),
        F.coalesce(
            F.col("discount_amount"),
            F.lit(0).cast("decimal(16,2)")
        ).alias("discount_amount"),
        F.coalesce(
            F.col("net_sales"),
            F.lit(0).cast("decimal(16,2)")
        ).alias("net_sales"),
        F.coalesce(
            F.col("cost_amount"),
            F.lit(0).cast("decimal(16,2)")
        ).alias("cost_amount"),
        F.coalesce(
            F.col("profit"),
            F.lit(0).cast("decimal(16,2)")
        ).alias("profit")
    )
)

print("Corrected fact_order created successfully.")


Corrected fact_order created successfully.


In [15]:
print(f"fact_order_detail rows: {fact_order_detail.count()}")
print(f"fact_order rows: {fact_order.count()}")

print("Null foreign-key counts in fact_order:")

fact_order.select([
    F.sum(
        F.when(F.col(column_name).isNull(), 1).otherwise(0)
    ).alias(column_name)
    for column_name in [
        "customer_key",
        "date_key",
        "order_status_key"
    ]
]).show()

fact_order.show(5, truncate=False)


fact_order_detail rows: 100000
fact_order rows: 50000
Null foreign-key counts in fact_order:
+------------+--------+----------------+
|customer_key|date_key|order_status_key|
+------------+--------+----------------+
|           0|       0|               0|
+------------+--------+----------------+

+--------+------------+-----------+--------+----------+----------------+------------+----------+-------------+--------------+-----------+---------------+---------+-----------+--------+
|order_id|customer_key|customer_id|date_key|order_date|order_status_key|order_status|line_count|product_count|total_quantity|gross_sales|discount_amount|net_sales|cost_amount|profit  |
+--------+------------+-----------+--------+----------+----------------+------------+----------+-------------+--------------+-----------+---------------+---------+-----------+--------+
|29      |873         |873        |20240519|2024-05-19|1               |CANCELLED   |3         |3            |13            |28868.35   |238.54   

In [16]:
# ============================================================
# STEP 9 - BUILD FACT_PAYMENT
# Grain: one row per payment transaction
# ============================================================

payments_base = (
    clean_dfs["payments"]
    .select(
        F.col("paymentid").alias("payment_id"),
        F.col("orderid").alias("order_id"),
        F.col("paymentmethod").alias("payment_method"),
        F.col("paymentdate").alias("payment_date"),
        F.col("amount")
    )
)

payment_order_lookup = (
    clean_dfs["orders"]
    .select(
        F.col("orderid").alias("order_id"),
        F.col("customerid").alias("customer_id")
    )
)

payment_customer_lookup = dim_customer.select(
    "customer_key",
    "customer_id"
)

payment_date_lookup = dim_date.select(
    "date_key",
    F.col("full_date").alias("payment_date")
)

payment_method_lookup = dim_payment_method.select(
    "payment_method_key",
    "payment_method"
)

fact_payment = (
    payments_base
    .join(
        payment_order_lookup,
        on="order_id",
        how="left"
    )
    .join(
        payment_customer_lookup,
        on="customer_id",
        how="left"
    )
    .join(
        payment_date_lookup,
        on="payment_date",
        how="left"
    )
    .join(
        payment_method_lookup,
        on="payment_method",
        how="left"
    )
    .select(
        "payment_id",
        "order_id",
        "customer_key",
        "customer_id",
        "date_key",
        "payment_date",
        "payment_method_key",
        "payment_method",
        "amount"
    )
)

print("fact_payment created successfully.")


fact_payment created successfully.


In [17]:
print(f"fact_payment rows: {fact_payment.count()}")

fact_payment.select([
    F.sum(
        F.when(F.col(column_name).isNull(), 1).otherwise(0)
    ).alias(column_name)
    for column_name in [
        "customer_key",
        "date_key",
        "payment_method_key"
    ]
]).show()

fact_payment.show(5, truncate=False)


fact_payment rows: 45000
+------------+--------+------------------+
|customer_key|date_key|payment_method_key|
+------------+--------+------------------+
|           0|       0|                 0|
+------------+--------+------------------+

+----------+--------+------------+-----------+--------+------------+------------------+--------------+-------+
|payment_id|order_id|customer_key|customer_id|date_key|payment_date|payment_method_key|payment_method|amount |
+----------+--------+------------+-----------+--------+------------+------------------+--------------+-------+
|1         |7144    |5945        |5945       |20260330|2026-03-30  |1                 |BANK TRANSFER |1912.52|
|3         |10641   |5977        |5977       |20250805|2025-08-05  |3                 |CREDIT CARD   |609.92 |
|5         |45202   |5532        |5532       |20240724|2024-07-24  |1                 |BANK TRANSFER |200.19 |
|10        |27807   |5615        |5615       |20251028|2025-10-28  |4                 |DEBIT 

In [18]:
# ============================================================
# STEP 10 - BUILD FACT_SHIPMENT
# Grain: one row per shipment
# ============================================================

shipments_base = (
    clean_dfs["shipments"]
    .select(
        F.col("shipmentid").alias("shipment_id"),
        F.col("orderid").alias("order_id"),
        F.col("shipperid").alias("shipper_id"),
        F.col("shipdate").alias("ship_date"),
        F.col("deliverydate").alias("delivery_date")
    )
)

shipment_order_lookup = (
    clean_dfs["orders"]
    .select(
        F.col("orderid").alias("order_id"),
        F.col("customerid").alias("customer_id")
    )
)

shipment_customer_lookup = dim_customer.select(
    "customer_key",
    "customer_id"
)

shipment_shipper_lookup = dim_shipper.select(
    "shipper_key",
    "shipper_id"
)

shipment_date_lookup = dim_date.select(
    "date_key",
    F.col("full_date").alias("ship_date")
)

fact_shipment = (
    shipments_base
    .join(
        shipment_order_lookup,
        on="order_id",
        how="left"
    )
    .join(
        shipment_customer_lookup,
        on="customer_id",
        how="left"
    )
    .join(
        shipment_shipper_lookup,
        on="shipper_id",
        how="left"
    )
    .join(
        shipment_date_lookup,
        on="ship_date",
        how="left"
    )
    .withColumn(
        "delivery_days",
        F.datediff(
            F.col("delivery_date"),
            F.col("ship_date")
        )
    )
    .select(
        "shipment_id",
        "order_id",
        "customer_key",
        "customer_id",
        "shipper_key",
        "shipper_id",
        "date_key",
        "ship_date",
        "delivery_date",
        "delivery_days"
    )
)

print("fact_shipment created successfully.")


fact_shipment created successfully.


In [19]:
print(f"fact_shipment rows: {fact_shipment.count()}")

fact_shipment.select([
    F.sum(
        F.when(F.col(column_name).isNull(), 1).otherwise(0)
    ).alias(column_name)
    for column_name in [
        "customer_key",
        "shipper_key",
        "date_key"
    ]
]).show()

fact_shipment.select(
    F.min("delivery_days").alias("min_delivery_days"),
    F.max("delivery_days").alias("max_delivery_days"),
    F.avg("delivery_days").alias("avg_delivery_days")
).show()

fact_shipment.show(5, truncate=False)


fact_shipment rows: 40000
+------------+-----------+--------+
|customer_key|shipper_key|date_key|
+------------+-----------+--------+
|           0|          0|       0|
+------------+-----------+--------+

+-----------------+-----------------+-----------------+
|min_delivery_days|max_delivery_days|avg_delivery_days|
+-----------------+-----------------+-----------------+
|                1|               10|         5.494075|
+-----------------+-----------------+-----------------+

+-----------+--------+------------+-----------+-----------+----------+--------+----------+-------------+-------------+
|shipment_id|order_id|customer_key|customer_id|shipper_key|shipper_id|date_key|ship_date |delivery_date|delivery_days|
+-----------+--------+------------+-----------+-----------+----------+--------+----------+-------------+-------------+
|12         |44150   |3755        |3755       |6          |6         |20250707|2025-07-07|2025-07-16   |9            |
|18         |14232   |9115        |9

In [21]:
# ============================================================
# BUILD AGGREGATED FACT TABLES
# ============================================================

# ------------------------------------------------------------
# fact_customer_sales
# Grain: one customer per day
# ------------------------------------------------------------

fact_customer_sales = (
    fact_order_detail
    .groupBy(
        "customer_key",
        "date_key"
    )
    .agg(
        F.countDistinct("order_id").alias("order_count"),
        F.sum("quantity").alias("quantity"),
        F.sum("net_sales")
         .cast("decimal(16,2)")
         .alias("sales"),
        F.sum("profit")
         .cast("decimal(16,2)")
         .alias("profit")
    )
)

print("fact_customer_sales created successfully.")


# ------------------------------------------------------------
# fact_product_sales
# Grain: one product per day
# ------------------------------------------------------------

fact_product_sales = (
    fact_order_detail
    .groupBy(
        "product_key",
        "date_key"
    )
    .agg(
        F.countDistinct("order_id").alias("order_count"),
        F.sum("quantity").alias("quantity_sold"),
        F.sum("gross_sales")
         .cast("decimal(16,2)")
         .alias("gross_sales"),
        F.sum("discount_amount")
         .cast("decimal(16,2)")
         .alias("discount"),
        F.sum("net_sales")
         .cast("decimal(16,2)")
         .alias("net_sales"),
        F.sum("cost_amount")
         .cast("decimal(16,2)")
         .alias("cost"),
        F.sum("profit")
         .cast("decimal(16,2)")
         .alias("profit")
    )
)

print("fact_product_sales created successfully.")


fact_customer_sales created successfully.
fact_product_sales created successfully.


In [22]:
print(f"fact_customer_sales rows: {fact_customer_sales.count()}")
print(f"fact_product_sales rows: {fact_product_sales.count()}")

print("\nCustomer sales sample:")
fact_customer_sales.show(5, truncate=False)

print("\nProduct sales sample:")
fact_product_sales.show(5, truncate=False)

print("\nCustomer sales null keys:")
fact_customer_sales.select([
    F.sum(
        F.when(F.col(column_name).isNull(), 1).otherwise(0)
    ).alias(column_name)
    for column_name in ["customer_key", "date_key"]
]).show()

print("\nProduct sales null keys:")
fact_product_sales.select([
    F.sum(
        F.when(F.col(column_name).isNull(), 1).otherwise(0)
    ).alias(column_name)
    for column_name in ["product_key", "date_key"]
]).show()


fact_customer_sales rows: 43036
fact_product_sales rows: 95102

Customer sales sample:
+------------+--------+-----------+--------+--------+--------+
|customer_key|date_key|order_count|quantity|sales   |profit  |
+------------+--------+-----------+--------+--------+--------+
|9477        |20240415|1          |1       |1775.47 |628.98  |
|7773        |20240829|1          |10      |8765.15 |3048.67 |
|5169        |20260712|1          |33      |45757.28|7585.54 |
|5246        |20240209|1          |14      |20951.14|4997.44 |
|105         |20250907|1          |22      |26592.21|11013.42|
+------------+--------+-----------+--------+--------+--------+
only showing top 5 rows


Product sales sample:
+-----------+--------+-----------+-------------+-----------+--------+---------+-------+-------+
|product_key|date_key|order_count|quantity_sold|gross_sales|discount|net_sales|cost   |profit |
+-----------+--------+-----------+-------------+-----------+--------+---------+-------+-------+
|673      

In [23]:
# ============================================================
# STEP 14 - FACT VALIDATION AND SALES RECONCILIATION
# ============================================================

print("Customer sales null keys:")
fact_customer_sales.select([
    F.sum(
        F.when(F.col(column_name).isNull(), 1).otherwise(0)
    ).alias(column_name)
    for column_name in ["customer_key", "date_key"]
]).show()

print("Product sales null keys:")
fact_product_sales.select([
    F.sum(
        F.when(F.col(column_name).isNull(), 1).otherwise(0)
    ).alias(column_name)
    for column_name in ["product_key", "date_key"]
]).show()


# ------------------------------------------------------------
# Compare sales across detailed and aggregated facts
# ------------------------------------------------------------

detail_totals = fact_order_detail.select(
    F.sum("gross_sales").alias("detail_gross_sales"),
    F.sum("discount_amount").alias("detail_discount"),
    F.sum("net_sales").alias("detail_net_sales"),
    F.sum("cost_amount").alias("detail_cost"),
    F.sum("profit").alias("detail_profit"),
    F.sum("quantity").alias("detail_quantity")
)

customer_totals = fact_customer_sales.select(
    F.sum("sales").alias("customer_net_sales"),
    F.sum("profit").alias("customer_profit"),
    F.sum("quantity").alias("customer_quantity")
)

product_totals = fact_product_sales.select(
    F.sum("gross_sales").alias("product_gross_sales"),
    F.sum("discount").alias("product_discount"),
    F.sum("net_sales").alias("product_net_sales"),
    F.sum("cost").alias("product_cost"),
    F.sum("profit").alias("product_profit"),
    F.sum("quantity_sold").alias("product_quantity")
)

print("Detailed fact totals:")
detail_totals.show()

print("Customer aggregated totals:")
customer_totals.show()

print("Product aggregated totals:")
product_totals.show()


Customer sales null keys:
+------------+--------+
|customer_key|date_key|
+------------+--------+
|           0|       0|
+------------+--------+

Product sales null keys:
+-----------+--------+
|product_key|date_key|
+-----------+--------+
|          0|       0|
+-----------+--------+

Detailed fact totals:
+------------------+---------------+----------------+------------+-------------+---------------+
|detail_gross_sales|detail_discount|detail_net_sales| detail_cost|detail_profit|detail_quantity|
+------------------+---------------+----------------+------------+-------------+---------------+
|      840507631.75|    59473211.30|    781034420.45|564111691.29| 216922729.16|         549473|
+------------------+---------------+----------------+------------+-------------+---------------+

Customer aggregated totals:
+------------------+---------------+-----------------+
|customer_net_sales|customer_profit|customer_quantity|
+------------------+---------------+-----------------+
|      7810

In [24]:
# ============================================================
# STEP 15 - WRITE DIMENSIONS AND FACTS TO S3 AS PARQUET
# ============================================================

PROCESSED_PATH = (
    "s3://mahmoud-sic-ecommerce-2026/"
    "processed/ecommerce"
)


def write_parquet(df, table_name):
    output_path = f"{PROCESSED_PATH}/{table_name}/"

    print(f"Writing {table_name} to:")
    print(output_path)

    (
        df.write
        .mode("overwrite")
        .format("parquet")
        .option("compression", "snappy")
        .save(output_path)
    )

    print(f"Finished: {table_name}")


# ------------------------------------------------------------
# Dimensions
# ------------------------------------------------------------

dimensions_to_write = {
    "dim_customer": dim_customer,
    "dim_product": dim_product,
    "dim_category": dim_category,
    "dim_department": dim_department,
    "dim_supplier": dim_supplier,
    "dim_employee": dim_employee,
    "dim_shipper": dim_shipper,
    "dim_date": dim_date,
    "dim_payment_method": dim_payment_method,
    "dim_order_status": dim_order_status
}


# ------------------------------------------------------------
# Facts
# ------------------------------------------------------------

facts_to_write = {
    "fact_order": fact_order,
    "fact_order_detail": fact_order_detail,
    "fact_payment": fact_payment,
    "fact_shipment": fact_shipment,
    "fact_customer_sales": fact_customer_sales,
    "fact_product_sales": fact_product_sales
}


# Write dimensions
for table_name, df in dimensions_to_write.items():
    write_parquet(df, table_name)


# Write facts
for table_name, df in facts_to_write.items():
    write_parquet(df, table_name)


print("All dimensions and facts were written successfully.")


Writing dim_customer to:
s3://mahmoud-sic-ecommerce-2026/processed/ecommerce/dim_customer/
Finished: dim_customer
Writing dim_product to:
s3://mahmoud-sic-ecommerce-2026/processed/ecommerce/dim_product/
Finished: dim_product
Writing dim_category to:
s3://mahmoud-sic-ecommerce-2026/processed/ecommerce/dim_category/
Finished: dim_category
Writing dim_department to:
s3://mahmoud-sic-ecommerce-2026/processed/ecommerce/dim_department/
Finished: dim_department
Writing dim_supplier to:
s3://mahmoud-sic-ecommerce-2026/processed/ecommerce/dim_supplier/
Finished: dim_supplier
Writing dim_employee to:
s3://mahmoud-sic-ecommerce-2026/processed/ecommerce/dim_employee/
Finished: dim_employee
Writing dim_shipper to:
s3://mahmoud-sic-ecommerce-2026/processed/ecommerce/dim_shipper/
Finished: dim_shipper
Writing dim_date to:
s3://mahmoud-sic-ecommerce-2026/processed/ecommerce/dim_date/
Finished: dim_date
Writing dim_payment_method to:
s3://mahmoud-sic-ecommerce-2026/processed/ecommerce/dim_payment_metho

In [25]:
# ============================================================
# STEP 15B - VERIFY PARQUET OUTPUTS
# ============================================================

all_outputs = {}

all_outputs.update(dimensions_to_write)
all_outputs.update(facts_to_write)

for table_name, df in all_outputs.items():
    output_path = f"{PROCESSED_PATH}/{table_name}/"

    parquet_check = spark.read.parquet(output_path)

    print(
        f"{table_name}: "
        f"memory_rows={df.count()}, "
        f"s3_rows={parquet_check.count()}, "
        f"columns={len(parquet_check.columns)}"
    )


dim_customer: memory_rows=10000, s3_rows=10000, columns=9
dim_product: memory_rows=1000, s3_rows=1000, columns=11
dim_category: memory_rows=20, s3_rows=20, columns=3
dim_department: memory_rows=10, s3_rows=10, columns=3
dim_supplier: memory_rows=100, s3_rows=100, columns=4
dim_employee: memory_rows=200, s3_rows=200, columns=9
dim_shipper: memory_rows=10, s3_rows=10, columns=3
dim_date: memory_rows=999, s3_rows=999, columns=10
dim_payment_method: memory_rows=5, s3_rows=5, columns=2
dim_order_status: memory_rows=5, s3_rows=5, columns=2
fact_order: memory_rows=50000, s3_rows=50000, columns=15
fact_order_detail: memory_rows=100000, s3_rows=100000, columns=17
fact_payment: memory_rows=45000, s3_rows=45000, columns=9
fact_shipment: memory_rows=40000, s3_rows=40000, columns=10
fact_customer_sales: memory_rows=43036, s3_rows=43036, columns=6
fact_product_sales: memory_rows=95102, s3_rows=95102, columns=9


In [2]:
# ============================================================
# STEP 16 - READ INCREMENTAL ORDERS JSON
# ============================================================

from pyspark.sql import functions as F

BUCKET = "mahmoud-sic-ecommerce-2026"

API_RAW_PATH = (
    f"s3://{BUCKET}/api_raw/orders/"
)

api_orders = (
    spark.read
    .option("multiLine", True)
    .json(API_RAW_PATH)
)

print(f"API JSON order events: {api_orders.count()}")

print("\nAPI JSON schema:")
api_orders.printSchema()

print("\nAPI JSON sample:")
api_orders.show(3, truncate=False)


API JSON order events: 3

API JSON schema:
root
 |-- metadata: struct (nullable = true)
 |    |-- event_id: string (nullable = true)
 |    |-- event_timestamp: string (nullable = true)
 |    |-- event_type: string (nullable = true)
 |    |-- source: string (nullable = true)
 |-- order: struct (nullable = true)
 |    |-- CustomerID: long (nullable = true)
 |    |-- OrderDate: string (nullable = true)
 |    |-- OrderID: long (nullable = true)
 |    |-- Status: string (nullable = true)
 |-- order_details: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- Discount: double (nullable = true)
 |    |    |-- OrderDetailID: long (nullable = true)
 |    |    |-- OrderID: long (nullable = true)
 |    |    |-- ProductID: long (nullable = true)
 |    |    |-- Quantity: long (nullable = true)
 |    |    |-- UnitPrice: double (nullable = true)
 |-- payment: struct (nullable = true)
 |    |-- Amount: double (nullable = true)
 |    |-- OrderID: long (nullable = true

In [3]:
# ============================================================
# STEP 17 - FLATTEN INCREMENTAL JSON
# ============================================================

from pyspark.sql import functions as F


# ------------------------------------------------------------
# 1. Orders
# Grain: one row per API order event
# ------------------------------------------------------------

api_orders_df = (
    api_orders
    .select(
        F.col("metadata.event_id").alias("event_id"),
        F.to_timestamp(
            F.col("metadata.event_timestamp")
        ).alias("event_timestamp"),
        F.col("metadata.event_type").alias("event_type"),
        F.col("metadata.source").alias("source"),

        F.col("order.OrderID")
         .cast("long")
         .alias("order_id"),

        F.col("order.CustomerID")
         .cast("long")
         .alias("customer_id"),

        F.to_date(
            F.col("order.OrderDate")
        ).alias("order_date"),

        F.upper(
            F.trim(F.col("order.Status"))
        ).alias("order_status")
    )
    .dropDuplicates(["order_id"])
)


# ------------------------------------------------------------
# 2. Order Details
# Grain: one row per product line in an API order
# ------------------------------------------------------------

api_order_details_df = (
    api_orders
    .select(
        F.col("order.OrderID")
         .cast("long")
         .alias("order_id"),

        F.explode_outer(
            F.col("order_details")
        ).alias("detail")
    )
    .select(
        "order_id",

        F.col("detail.OrderDetailID")
         .cast("long")
         .alias("order_detail_id"),

        F.col("detail.ProductID")
         .cast("long")
         .alias("product_id"),

        F.col("detail.Quantity")
         .cast("int")
         .alias("quantity"),

        F.col("detail.UnitPrice")
         .cast("decimal(12,2)")
         .alias("unit_price"),

        F.col("detail.Discount")
         .cast("decimal(5,2)")
         .alias("discount")
    )
    .dropDuplicates(["order_detail_id"])
)


# ------------------------------------------------------------
# 3. Payments
# Grain: one row per API payment
# ------------------------------------------------------------

api_payments_df = (
    api_orders
    .select(
        F.col("payment.PaymentID")
         .cast("long")
         .alias("payment_id"),

        F.col("payment.OrderID")
         .cast("long")
         .alias("order_id"),

        F.upper(
            F.trim(F.col("payment.PaymentMethod"))
        ).alias("payment_method"),

        F.to_date(
            F.col("payment.PaymentDate")
        ).alias("payment_date"),

        F.col("payment.Amount")
         .cast("decimal(14,2)")
         .alias("amount"),

        F.upper(
            F.trim(F.col("payment.PaymentStatus"))
        ).alias("payment_status")
    )
    .filter(F.col("payment_id").isNotNull())
    .dropDuplicates(["payment_id"])
)


# ------------------------------------------------------------
# 4. Shipments
# Grain: one row per API shipment
# ------------------------------------------------------------

api_shipments_df = (
    api_orders
    .select(
        F.col("shipment.ShipmentID")
         .cast("long")
         .alias("shipment_id"),

        F.col("shipment.OrderID")
         .cast("long")
         .alias("order_id"),

        F.col("shipment.ShipperID")
         .cast("long")
         .alias("shipper_id"),

        F.to_date(
            F.col("shipment.ShipDate")
        ).alias("ship_date"),

        F.to_date(
            F.col("shipment.DeliveryDate")
        ).alias("delivery_date")
    )
    .filter(F.col("shipment_id").isNotNull())
    .dropDuplicates(["shipment_id"])
)


print("Incremental JSON was flattened successfully.")


Incremental JSON was flattened successfully.


In [4]:
# ============================================================
# STEP 17B - VERIFY FLATTENED API DATA
# ============================================================

print(f"API orders: {api_orders_df.count()}")
print(f"API order details: {api_order_details_df.count()}")
print(f"API payments: {api_payments_df.count()}")
print(f"API shipments: {api_shipments_df.count()}")

print("\nOrder ID range:")
api_orders_df.select(
    F.min("order_id").alias("min_order_id"),
    F.max("order_id").alias("max_order_id"),
    F.countDistinct("order_id").alias("distinct_order_ids")
).show()

print("\nOrder detail ID range:")
api_order_details_df.select(
    F.min("order_detail_id").alias("min_order_detail_id"),
    F.max("order_detail_id").alias("max_order_detail_id"),
    F.countDistinct("order_detail_id").alias("distinct_order_detail_ids")
).show()

print("\nPayment ID range:")
api_payments_df.select(
    F.min("payment_id").alias("min_payment_id"),
    F.max("payment_id").alias("max_payment_id"),
    F.countDistinct("payment_id").alias("distinct_payment_ids")
).show()

print("\nShipment ID range:")
api_shipments_df.select(
    F.min("shipment_id").alias("min_shipment_id"),
    F.max("shipment_id").alias("max_shipment_id"),
    F.countDistinct("shipment_id").alias("distinct_shipment_ids")
).show()


API orders: 3
API order details: 10
API payments: 3
API shipments: 3

Order ID range:
+------------+------------+------------------+
|min_order_id|max_order_id|distinct_order_ids|
+------------+------------+------------------+
|       50001|       50003|                 3|
+------------+------------+------------------+


Order detail ID range:
+-------------------+-------------------+-------------------------+
|min_order_detail_id|max_order_detail_id|distinct_order_detail_ids|
+-------------------+-------------------+-------------------------+
|             100001|             100010|                       10|
+-------------------+-------------------+-------------------------+


Payment ID range:
+--------------+--------------+--------------------+
|min_payment_id|max_payment_id|distinct_payment_ids|
+--------------+--------------+--------------------+
|         45001|         45003|                   3|
+--------------+--------------+--------------------+


Shipment ID range:
+-------

In [11]:
# ============================================================
# STEP 18 - VALIDATE INCREMENTAL API DATA
# ============================================================

# Existing source keys
customer_keys = clean_dfs["customers"].select(
    F.col("customerid").alias("customer_id")
).distinct()

product_keys = clean_dfs["products"].select(
    F.col("productid").alias("product_id")
).distinct()

shipper_keys = clean_dfs["shippers"].select(
    F.col("shipperid").alias("shipper_id")
).distinct()

historical_order_keys = clean_dfs["orders"].select(
    F.col("orderid").alias("order_id")
).distinct()


print("API order statuses:")
api_orders_df.groupBy("order_status").count().show(
    truncate=False
)

print("API payment methods:")
api_payments_df.groupBy("payment_method").count().show(
    truncate=False
)


# Check API customer references
invalid_api_customers = (
    api_orders_df
    .select("order_id", "customer_id")
    .join(customer_keys, on="customer_id", how="left_anti")
)

print(
    "API orders with nonexistent customers:",
    invalid_api_customers.count()
)


# Check API product references
invalid_api_products = (
    api_order_details_df
    .select("order_detail_id", "product_id")
    .join(product_keys, on="product_id", how="left_anti")
)

print(
    "API order details with nonexistent products:",
    invalid_api_products.count()
)


# Check API shipper references
invalid_api_shippers = (
    api_shipments_df
    .select("shipment_id", "shipper_id")
    .join(shipper_keys, on="shipper_id", how="left_anti")
)

print(
    "API shipments with nonexistent shippers:",
    invalid_api_shippers.count()
)


# Check overlap with historical orders
overlapping_orders = (
    api_orders_df
    .select("order_id")
    .join(historical_order_keys, on="order_id", how="inner")
)

print(
    "API orders already existing historically:",
    overlapping_orders.count()
)


# Check dimensions for statuses and payment methods
print("API statuses missing from dim_order_status:")

api_orders_df.select("order_status").distinct().join(
    dim_order_status.select("order_status"),
    on="order_status",
    how="left_anti"
).show(truncate=False)


print("API payment methods missing from dim_payment_method:")

api_payments_df.select("payment_method").distinct().join(
    dim_payment_method.select("payment_method"),
    on="payment_method",
    how="left_anti"
).show(truncate=False)


API order statuses:
+------------+-----+
|order_status|count|
+------------+-----+
|CONFIRMED   |3    |
+------------+-----+

API payment methods:
+----------------+-----+
|payment_method  |count|
+----------------+-----+
|PAYPAL          |1    |
|CASH ON DELIVERY|2    |
+----------------+-----+

API orders with nonexistent customers: 0
API order details with nonexistent products: 0
API shipments with nonexistent shippers: 0
API orders already existing historically: 0
API statuses missing from dim_order_status:
+------------+
|order_status|
+------------+
|CONFIRMED   |
+------------+

API payment methods missing from dim_payment_method:
+----------------+
|payment_method  |
+----------------+
|CASH ON DELIVERY|
+----------------+


In [12]:
# ============================================================
# STEP 19 - EXTEND DIMENSIONS FOR NEW API VALUES
# ============================================================

# ------------------------------------------------------------
# Add new order statuses while preserving existing keys
# ------------------------------------------------------------

existing_statuses = dim_order_status.select(
    "order_status_key",
    "order_status"
)

new_api_statuses = (
    api_orders_df
    .select("order_status")
    .filter(F.col("order_status").isNotNull())
    .dropDuplicates()
    .join(
        existing_statuses.select("order_status"),
        on="order_status",
        how="left_anti"
    )
)

max_status_key = (
    existing_statuses
    .agg(F.max("order_status_key").alias("max_key"))
    .first()["max_key"]
)

status_window = Window.orderBy("order_status")

new_api_statuses = (
    new_api_statuses
    .withColumn(
        "order_status_key",
        F.row_number().over(status_window) + F.lit(max_status_key)
    )
    .select(
        "order_status_key",
        "order_status"
    )
)

dim_order_status = (
    existing_statuses
    .unionByName(new_api_statuses)
    .orderBy("order_status_key")
)


# ------------------------------------------------------------
# Add new payment methods while preserving existing keys
# ------------------------------------------------------------

existing_payment_methods = dim_payment_method.select(
    "payment_method_key",
    "payment_method"
)

new_api_payment_methods = (
    api_payments_df
    .select("payment_method")
    .filter(F.col("payment_method").isNotNull())
    .dropDuplicates()
    .join(
        existing_payment_methods.select("payment_method"),
        on="payment_method",
        how="left_anti"
    )
)

max_payment_key = (
    existing_payment_methods
    .agg(F.max("payment_method_key").alias("max_key"))
    .first()["max_key"]
)

payment_window = Window.orderBy("payment_method")

new_api_payment_methods = (
    new_api_payment_methods
    .withColumn(
        "payment_method_key",
        F.row_number().over(payment_window) + F.lit(max_payment_key)
    )
    .select(
        "payment_method_key",
        "payment_method"
    )
)

dim_payment_method = (
    existing_payment_methods
    .unionByName(new_api_payment_methods)
    .orderBy("payment_method_key")
)


print("Dimensions extended successfully.")


Dimensions extended successfully.


In [13]:
print("Updated order status dimension:")
dim_order_status.orderBy("order_status_key").show(
    truncate=False
)

print("Updated payment method dimension:")
dim_payment_method.orderBy("payment_method_key").show(
    truncate=False
)

print("Missing API statuses after update:")
api_orders_df.select("order_status").distinct().join(
    dim_order_status.select("order_status"),
    on="order_status",
    how="left_anti"
).show()

print("Missing API payment methods after update:")
api_payments_df.select("payment_method").distinct().join(
    dim_payment_method.select("payment_method"),
    on="payment_method",
    how="left_anti"
).show()


Updated order status dimension:
+----------------+------------+
|order_status_key|order_status|
+----------------+------------+
|1               |CANCELLED   |
|2               |DELIVERED   |
|3               |PENDING     |
|4               |PROCESSING  |
|5               |SHIPPED     |
|6               |CONFIRMED   |
+----------------+------------+

Updated payment method dimension:
+------------------+----------------+
|payment_method_key|payment_method  |
+------------------+----------------+
|1                 |BANK TRANSFER   |
|2                 |CASH            |
|3                 |CREDIT CARD     |
|4                 |DEBIT CARD      |
|5                 |PAYPAL          |
|6                 |CASH ON DELIVERY|
+------------------+----------------+

Missing API statuses after update:
+------------+
|order_status|
+------------+
+------------+

Missing API payment methods after update:
+--------------+
|payment_method|
+--------------+
+--------------+


In [16]:
# ============================================================
# HELPER - WRITE PARQUET TO S3
# ============================================================

PROCESSED_PATH = (
    "s3://mahmoud-sic-ecommerce-2026/"
    "processed/ecommerce"
)


def write_parquet(df, table_name):
    output_path = f"{PROCESSED_PATH}/{table_name}/"

    print(f"Writing {table_name} to {output_path}")

    (
        df.write
        .mode("overwrite")
        .format("parquet")
        .option("compression", "snappy")
        .save(output_path)
    )

    print(f"Finished: {table_name}")


In [17]:
# ============================================================
# SAVE UPDATED DIMENSIONS
# ============================================================

write_parquet(dim_order_status, "dim_order_status")
write_parquet(dim_payment_method, "dim_payment_method")

print("Updated dimensions saved to S3.")


Writing dim_order_status to s3://mahmoud-sic-ecommerce-2026/processed/ecommerce/dim_order_status/
Finished: dim_order_status
Writing dim_payment_method to s3://mahmoud-sic-ecommerce-2026/processed/ecommerce/dim_payment_method/
Finished: dim_payment_method
Updated dimensions saved to S3.


In [18]:
# ============================================================
# STEP 21 - BUILD INCREMENTAL FACT_ORDER_DETAIL
# Grain: one row per product line in a new API order
# ============================================================

api_orders_for_fact = (
    api_orders_df
    .select(
        "order_id",
        "customer_id",
        "order_date",
        "order_status"
    )
)

api_product_lookup = dim_product.select(
    "product_key",
    "product_id",
    "cost"
)

api_customer_lookup = dim_customer.select(
    "customer_key",
    "customer_id"
)

api_date_lookup = dim_date.select(
    "date_key",
    F.col("full_date").alias("order_date")
)

api_status_lookup = dim_order_status.select(
    "order_status_key",
    "order_status"
)

api_fact_order_detail = (
    api_order_details_df
    .join(
        api_orders_for_fact,
        on="order_id",
        how="left"
    )
    .join(
        api_product_lookup,
        on="product_id",
        how="left"
    )
    .join(
        api_customer_lookup,
        on="customer_id",
        how="left"
    )
    .join(
        api_date_lookup,
        on="order_date",
        how="left"
    )
    .join(
        api_status_lookup,
        on="order_status",
        how="left"
    )
    .withColumn(
        "gross_sales",
        (
            F.col("quantity") * F.col("unit_price")
        ).cast("decimal(16,2)")
    )
    .withColumn(
        "discount_amount",
        (
            F.col("quantity")
            * F.col("unit_price")
            * F.col("discount")
            / F.lit(100)
        ).cast("decimal(16,2)")
    )
    .withColumn(
        "net_sales",
        (
            F.col("gross_sales")
            - F.col("discount_amount")
        ).cast("decimal(16,2)")
    )
    .withColumn(
        "cost_amount",
        (
            F.col("quantity") * F.col("cost")
        ).cast("decimal(16,2)")
    )
    .withColumn(
        "profit",
        (
            F.col("net_sales")
            - F.col("cost_amount")
        ).cast("decimal(16,2)")
    )
    .select(
        "order_detail_id",
        "order_id",
        "product_key",
        "product_id",
        "customer_key",
        "customer_id",
        "date_key",
        "order_date",
        "order_status_key",
        "quantity",
        "unit_price",
        "discount",
        "gross_sales",
        "discount_amount",
        "net_sales",
        "cost_amount",
        "profit"
    )
    .dropDuplicates(["order_detail_id"])
)

print(
    "Incremental fact_order_detail rows:",
    api_fact_order_detail.count()
)


Incremental fact_order_detail rows: 10


In [19]:
api_fact_order_detail.select([
    F.sum(
        F.when(F.col(column_name).isNull(), 1).otherwise(0)
    ).alias(column_name)
    for column_name in [
        "product_key",
        "customer_key",
        "date_key",
        "order_status_key"
    ]
]).show()


+-----------+------------+--------+----------------+
|product_key|customer_key|date_key|order_status_key|
+-----------+------------+--------+----------------+
|          0|           0|       0|               0|
+-----------+------------+--------+----------------+


In [20]:
# ============================================================
# STEP 22 - BUILD INCREMENTAL FACT_ORDER
# Grain: one row per new API order
# ============================================================

api_order_metrics = (
    api_fact_order_detail
    .groupBy("order_id")
    .agg(
        F.countDistinct("order_detail_id").alias("line_count"),
        F.countDistinct("product_id").alias("product_count"),
        F.sum("quantity").alias("total_quantity"),
        F.sum("gross_sales")
         .cast("decimal(16,2)")
         .alias("gross_sales"),
        F.sum("discount_amount")
         .cast("decimal(16,2)")
         .alias("discount_amount"),
        F.sum("net_sales")
         .cast("decimal(16,2)")
         .alias("net_sales"),
        F.sum("cost_amount")
         .cast("decimal(16,2)")
         .alias("cost_amount"),
        F.sum("profit")
         .cast("decimal(16,2)")
         .alias("profit")
    )
)

api_fact_order = (
    api_orders_for_fact
    .join(
        dim_customer.select(
            "customer_key",
            "customer_id"
        ),
        on="customer_id",
        how="left"
    )
    .join(
        dim_date.select(
            "date_key",
            F.col("full_date").alias("order_date")
        ),
        on="order_date",
        how="left"
    )
    .join(
        dim_order_status.select(
            "order_status_key",
            "order_status"
        ),
        on="order_status",
        how="left"
    )
    .join(
        api_order_metrics,
        on="order_id",
        how="left"
    )
    .select(
        "order_id",
        "customer_key",
        "customer_id",
        "date_key",
        "order_date",
        "order_status_key",
        "order_status",
        F.coalesce(F.col("line_count"), F.lit(0)).alias("line_count"),
        F.coalesce(F.col("product_count"), F.lit(0)).alias("product_count"),
        F.coalesce(F.col("total_quantity"), F.lit(0)).alias("total_quantity"),
        F.coalesce(
            F.col("gross_sales"),
            F.lit(0).cast("decimal(16,2)")
        ).alias("gross_sales"),
        F.coalesce(
            F.col("discount_amount"),
            F.lit(0).cast("decimal(16,2)")
        ).alias("discount_amount"),
        F.coalesce(
            F.col("net_sales"),
            F.lit(0).cast("decimal(16,2)")
        ).alias("net_sales"),
        F.coalesce(
            F.col("cost_amount"),
            F.lit(0).cast("decimal(16,2)")
        ).alias("cost_amount"),
        F.coalesce(
            F.col("profit"),
            F.lit(0).cast("decimal(16,2)")
        ).alias("profit")
    )
)

print(
    "Incremental fact_order rows:",
    api_fact_order.count()
)

api_fact_order.select(
    F.min("order_id").alias("min_order_id"),
    F.max("order_id").alias("max_order_id"),
    F.countDistinct("order_id").alias("distinct_order_ids")
).show()


Incremental fact_order rows: 3
+------------+------------+------------------+
|min_order_id|max_order_id|distinct_order_ids|
+------------+------------+------------------+
|       50001|       50003|                 3|
+------------+------------+------------------+


In [21]:
# ============================================================
# STEP 23 - BUILD INCREMENTAL PAYMENT AND SHIPMENT FACTS
# ============================================================

api_fact_payment = (
    api_payments_df
    .join(
        clean_dfs["orders"].select(
            F.col("orderid").alias("order_id"),
            F.col("customerid").alias("customer_id")
        ),
        on="order_id",
        how="left"
    )
    .join(
        api_customer_lookup,
        on="customer_id",
        how="left"
    )
    .join(
        dim_date.select(
            "date_key",
            F.col("full_date").alias("payment_date")
        ),
        on="payment_date",
        how="left"
    )
    .join(
        dim_payment_method.select(
            "payment_method_key",
            "payment_method"
        ),
        on="payment_method",
        how="left"
    )
    .select(
        "payment_id",
        "order_id",
        "customer_key",
        "customer_id",
        "date_key",
        "payment_date",
        "payment_method_key",
        "payment_method",
        "amount"
    )
)

# For API orders, use the API orders themselves for customer lookup
api_fact_payment = (
    api_payments_df
    .join(
        api_orders_for_fact.select(
            "order_id",
            "customer_id"
        ),
        on="order_id",
        how="left"
    )
    .join(
        api_customer_lookup,
        on="customer_id",
        how="left"
    )
    .join(
        dim_date.select(
            "date_key",
            F.col("full_date").alias("payment_date")
        ),
        on="payment_date",
        how="left"
    )
    .join(
        dim_payment_method.select(
            "payment_method_key",
            "payment_method"
        ),
        on="payment_method",
        how="left"
    )
    .select(
        "payment_id",
        "order_id",
        "customer_key",
        "customer_id",
        "date_key",
        "payment_date",
        "payment_method_key",
        "payment_method",
        "amount"
    )
)

api_fact_shipment = (
    api_shipments_df
    .join(
        api_orders_for_fact.select(
            "order_id",
            "customer_id"
        ),
        on="order_id",
        how="left"
    )
    .join(
        api_customer_lookup,
        on="customer_id",
        how="left"
    )
    .join(
        dim_shipper.select(
            "shipper_key",
            "shipper_id"
        ),
        on="shipper_id",
        how="left"
    )
    .join(
        dim_date.select(
            "date_key",
            F.col("full_date").alias("ship_date")
        ),
        on="ship_date",
        how="left"
    )
    .withColumn(
        "delivery_days",
        F.datediff(
            F.col("delivery_date"),
            F.col("ship_date")
        )
    )
    .select(
        "shipment_id",
        "order_id",
        "customer_key",
        "customer_id",
        "shipper_key",
        "shipper_id",
        "date_key",
        "ship_date",
        "delivery_date",
        "delivery_days"
    )
)

print("Incremental payment rows:", api_fact_payment.count())
print("Incremental shipment rows:", api_fact_shipment.count())


Incremental payment rows: 3
Incremental shipment rows: 3


In [22]:
# ============================================================
# STEP 24 - WRITE INCREMENTAL TRANSACTION FACTS
# ============================================================

incremental_transaction_facts = {
    "fact_order_detail": api_fact_order_detail,
    "fact_order": api_fact_order,
    "fact_payment": api_fact_payment,
    "fact_shipment": api_fact_shipment
}

for table_name, df in incremental_transaction_facts.items():

    output_path = f"{PROCESSED_PATH}/{table_name}/"

    print(
        f"Appending {df.count()} rows to {output_path}"
    )

    (
        df.write
        .mode("append")
        .format("parquet")
        .option("compression", "snappy")
        .save(output_path)
    )

    print(f"Finished appending {table_name}")

print("Incremental transaction facts saved successfully.")


Appending 10 rows to s3://mahmoud-sic-ecommerce-2026/processed/ecommerce/fact_order_detail/
Finished appending fact_order_detail
Appending 3 rows to s3://mahmoud-sic-ecommerce-2026/processed/ecommerce/fact_order/
Finished appending fact_order
Appending 3 rows to s3://mahmoud-sic-ecommerce-2026/processed/ecommerce/fact_payment/
Finished appending fact_payment
Appending 3 rows to s3://mahmoud-sic-ecommerce-2026/processed/ecommerce/fact_shipment/
Finished appending fact_shipment
Incremental transaction facts saved successfully.


In [23]:
# ============================================================
# STEP 24B - VERIFY INCREMENTAL TRANSACTION FACTS
# ============================================================

for table_name in incremental_transaction_facts.keys():

    output_path = f"{PROCESSED_PATH}/{table_name}/"

    saved_df = spark.read.parquet(output_path)

    print(
        f"{table_name}: "
        f"{saved_df.count()} total rows after incremental load"
    )


fact_order_detail: 100010 total rows after incremental load
fact_order: 50003 total rows after incremental load
fact_payment: 45003 total rows after incremental load
fact_shipment: 40003 total rows after incremental load


In [24]:
# ============================================================
# STEP 25 - BUILD INCREMENTAL AGGREGATED FACTS
# ============================================================

api_fact_customer_sales = (
    api_fact_order_detail
    .groupBy(
        "customer_key",
        "date_key"
    )
    .agg(
        F.countDistinct("order_id").alias("order_count"),
        F.sum("quantity").alias("quantity"),
        F.sum("net_sales")
         .cast("decimal(16,2)")
         .alias("sales"),
        F.sum("profit")
         .cast("decimal(16,2)")
         .alias("profit")
    )
)

api_fact_product_sales = (
    api_fact_order_detail
    .groupBy(
        "product_key",
        "date_key"
    )
    .agg(
        F.countDistinct("order_id").alias("order_count"),
        F.sum("quantity").alias("quantity_sold"),
        F.sum("gross_sales")
         .cast("decimal(16,2)")
         .alias("gross_sales"),
        F.sum("discount_amount")
         .cast("decimal(16,2)")
         .alias("discount"),
        F.sum("net_sales")
         .cast("decimal(16,2)")
         .alias("net_sales"),
        F.sum("cost_amount")
         .cast("decimal(16,2)")
         .alias("cost"),
        F.sum("profit")
         .cast("decimal(16,2)")
         .alias("profit")
    )
)

print(
    "New customer-sales rows:",
    api_fact_customer_sales.count()
)

print(
    "New product-sales rows:",
    api_fact_product_sales.count()
)


New customer-sales rows: 3
New product-sales rows: 10


In [26]:
# ============================================================
# RESTORE FACT_ORDER_DETAIL FROM S3
# ============================================================

PROCESSED_PATH = (
    "s3://mahmoud-sic-ecommerce-2026/"
    "processed/ecommerce"
)

fact_order_detail = spark.read.parquet(
    f"{PROCESSED_PATH}/fact_order_detail/"
)

print(
    "Loaded fact_order_detail rows:",
    fact_order_detail.count()
)


Loaded fact_order_detail rows: 100010


In [27]:
# ============================================================
# REBUILD AGGREGATED FACTS FROM CURRENT S3 DATA
# ============================================================

fact_customer_sales_updated = (
    fact_order_detail
    .groupBy(
        "customer_key",
        "date_key"
    )
    .agg(
        F.countDistinct("order_id").alias("order_count"),
        F.sum("quantity").alias("quantity"),
        F.sum("net_sales")
         .cast("decimal(16,2)")
         .alias("sales"),
        F.sum("profit")
         .cast("decimal(16,2)")
         .alias("profit")
    )
)

fact_product_sales_updated = (
    fact_order_detail
    .groupBy(
        "product_key",
        "date_key"
    )
    .agg(
        F.countDistinct("order_id").alias("order_count"),
        F.sum("quantity").alias("quantity_sold"),
        F.sum("gross_sales")
         .cast("decimal(16,2)")
         .alias("gross_sales"),
        F.sum("discount_amount")
         .cast("decimal(16,2)")
         .alias("discount"),
        F.sum("net_sales")
         .cast("decimal(16,2)")
         .alias("net_sales"),
        F.sum("cost_amount")
         .cast("decimal(16,2)")
         .alias("cost"),
        F.sum("profit")
         .cast("decimal(16,2)")
         .alias("profit")
    )
)

print(
    "Updated customer-sales rows:",
    fact_customer_sales_updated.count()
)

print(
    "Updated product-sales rows:",
    fact_product_sales_updated.count()
)


Updated customer-sales rows: 43039
Updated product-sales rows: 95112


In [28]:
# ============================================================
# RECONCILE UPDATED AGGREGATED FACTS
# ============================================================

print("Customer sales totals:")
fact_customer_sales_updated.select(
    F.sum("sales").alias("sales"),
    F.sum("profit").alias("profit"),
    F.sum("quantity").alias("quantity")
).show()

print("Product sales totals:")
fact_product_sales_updated.select(
    F.sum("net_sales").alias("net_sales"),
    F.sum("profit").alias("profit"),
    F.sum("quantity_sold").alias("quantity_sold")
).show()

print("Detail fact totals:")
fact_order_detail.select(
    F.sum("net_sales").alias("net_sales"),
    F.sum("profit").alias("profit"),
    F.sum("quantity").alias("quantity")
).show()


Customer sales totals:
+------------+------------+--------+
|       sales|      profit|quantity|
+------------+------------+--------+
|781050389.06|216904417.76|  549510|
+------------+------------+--------+

Product sales totals:
+------------+------------+-------------+
|   net_sales|      profit|quantity_sold|
+------------+------------+-------------+
|781050389.06|216904417.76|       549510|
+------------+------------+-------------+

Detail fact totals:
+------------+------------+--------+
|   net_sales|      profit|quantity|
+------------+------------+--------+
|781050389.06|216904417.76|  549510|
+------------+------------+--------+


In [29]:
# ============================================================
# SAVE UPDATED AGGREGATED FACTS
# ============================================================

fact_customer_sales_updated.write \
    .mode("overwrite") \
    .format("parquet") \
    .option("compression", "snappy") \
    .save(f"{PROCESSED_PATH}/fact_customer_sales/")

fact_product_sales_updated.write \
    .mode("overwrite") \
    .format("parquet") \
    .option("compression", "snappy") \
    .save(f"{PROCESSED_PATH}/fact_product_sales/")

print("Aggregated facts updated successfully.")


Aggregated facts updated successfully.


In [30]:
# ============================================================
# STEP 26 - FINAL S3 INCREMENTAL VALIDATION
# ============================================================

PROCESSED_PATH = (
    "s3://mahmoud-sic-ecommerce-2026/"
    "processed/ecommerce"
)

expected_tables = {
    "dim_customer": 10000,
    "dim_product": 1000,
    "dim_category": 20,
    "dim_department": 10,
    "dim_supplier": 100,
    "dim_employee": 200,
    "dim_shipper": 10,
    "dim_date": 999,
    "dim_payment_method": 6,
    "dim_order_status": 6,
    "fact_order": 50003,
    "fact_order_detail": 100010,
    "fact_payment": 45003,
    "fact_shipment": 40003,
    "fact_customer_sales": 43039,
    "fact_product_sales": 95112
}

for table_name, expected_count in expected_tables.items():

    path = f"{PROCESSED_PATH}/{table_name}/"

    saved_df = spark.read.parquet(path)
    actual_count = saved_df.count()

    status = "PASS" if actual_count == expected_count else "CHECK"

    print(
        f"{status} | {table_name} | "
        f"expected={expected_count} | actual={actual_count}"
    )


PASS | dim_customer | expected=10000 | actual=10000
PASS | dim_product | expected=1000 | actual=1000
PASS | dim_category | expected=20 | actual=20
PASS | dim_department | expected=10 | actual=10
PASS | dim_supplier | expected=100 | actual=100
PASS | dim_employee | expected=200 | actual=200
PASS | dim_shipper | expected=10 | actual=10
PASS | dim_date | expected=999 | actual=999
PASS | dim_payment_method | expected=6 | actual=6
PASS | dim_order_status | expected=6 | actual=6
PASS | fact_order | expected=50003 | actual=50003
PASS | fact_order_detail | expected=100010 | actual=100010
PASS | fact_payment | expected=45003 | actual=45003
PASS | fact_shipment | expected=40003 | actual=40003
PASS | fact_customer_sales | expected=43039 | actual=43039
PASS | fact_product_sales | expected=95112 | actual=95112


In [31]:
# ============================================================
# DUPLICATE KEY VALIDATION
# ============================================================

key_checks = {
    "dim_customer": ["customer_key"],
    "dim_product": ["product_key"],
    "dim_category": ["category_key"],
    "dim_department": ["department_key"],
    "dim_supplier": ["supplier_key"],
    "dim_employee": ["employee_key"],
    "dim_shipper": ["shipper_key"],
    "dim_date": ["date_key"],
    "dim_payment_method": ["payment_method_key"],
    "dim_order_status": ["order_status_key"],
    "fact_order": ["order_id"],
    "fact_order_detail": ["order_detail_id"],
    "fact_payment": ["payment_id"],
    "fact_shipment": ["shipment_id"],
    "fact_customer_sales": ["customer_key", "date_key"],
    "fact_product_sales": ["product_key", "date_key"]
}

for table_name, key_columns in key_checks.items():

    df = spark.read.parquet(
        f"{PROCESSED_PATH}/{table_name}/"
    )

    duplicate_groups = (
        df.groupBy(*key_columns)
          .count()
          .filter(F.col("count") > 1)
          .count()
    )

    status = "PASS" if duplicate_groups == 0 else "CHECK"

    print(
        f"{status} | {table_name} | "
        f"duplicate_key_groups={duplicate_groups}"
    )


PASS | dim_customer | duplicate_key_groups=0
PASS | dim_product | duplicate_key_groups=0
PASS | dim_category | duplicate_key_groups=0
PASS | dim_department | duplicate_key_groups=0
PASS | dim_supplier | duplicate_key_groups=0
PASS | dim_employee | duplicate_key_groups=0
PASS | dim_shipper | duplicate_key_groups=0
PASS | dim_date | duplicate_key_groups=0
PASS | dim_payment_method | duplicate_key_groups=0
PASS | dim_order_status | duplicate_key_groups=0
PASS | fact_order | duplicate_key_groups=0
PASS | fact_order_detail | duplicate_key_groups=0
PASS | fact_payment | duplicate_key_groups=0
PASS | fact_shipment | duplicate_key_groups=0
PASS | fact_customer_sales | duplicate_key_groups=0
PASS | fact_product_sales | duplicate_key_groups=0


In [32]:
# ============================================================
# FINAL ROW COUNT VALIDATION
# ============================================================

expected_tables = {
    "dim_customer": 10000,
    "dim_product": 1000,
    "dim_category": 20,
    "dim_department": 10,
    "dim_supplier": 100,
    "dim_employee": 200,
    "dim_shipper": 10,
    "dim_date": 999,
    "dim_payment_method": 6,
    "dim_order_status": 6,
    "fact_order": 50003,
    "fact_order_detail": 100010,
    "fact_payment": 45003,
    "fact_shipment": 40003,
    "fact_customer_sales": 43039,
    "fact_product_sales": 95112
}

for table_name, expected_count in expected_tables.items():
    saved_df = spark.read.parquet(
        f"{PROCESSED_PATH}/{table_name}/"
    )

    actual_count = saved_df.count()

    status = "PASS" if actual_count == expected_count else "CHECK"

    print(
        f"{status} | {table_name} | "
        f"expected={expected_count} | actual={actual_count}"
    )


PASS | dim_customer | expected=10000 | actual=10000
PASS | dim_product | expected=1000 | actual=1000
PASS | dim_category | expected=20 | actual=20
PASS | dim_department | expected=10 | actual=10
PASS | dim_supplier | expected=100 | actual=100
PASS | dim_employee | expected=200 | actual=200
PASS | dim_shipper | expected=10 | actual=10
PASS | dim_date | expected=999 | actual=999
PASS | dim_payment_method | expected=6 | actual=6
PASS | dim_order_status | expected=6 | actual=6
PASS | fact_order | expected=50003 | actual=50003
PASS | fact_order_detail | expected=100010 | actual=100010
PASS | fact_payment | expected=45003 | actual=45003
PASS | fact_shipment | expected=40003 | actual=40003
PASS | fact_customer_sales | expected=43039 | actual=43039
PASS | fact_product_sales | expected=95112 | actual=95112


In [33]:
# ============================================================
# API RECORD VALIDATION
# ============================================================

checks = {
    "fact_order": ("order_id", 50001, 50003),
    "fact_order_detail": ("order_detail_id", 100001, 100010),
    "fact_payment": ("payment_id", 45001, 45003),
    "fact_shipment": ("shipment_id", 40001, 40003)
}

for table_name, (key_column, min_value, max_value) in checks.items():

    df = spark.read.parquet(
        f"{PROCESSED_PATH}/{table_name}/"
    )

    new_rows = df.filter(
        (F.col(key_column) >= min_value)
        & (F.col(key_column) <= max_value)
    ).count()

    print(
        f"{table_name}: new_rows_found={new_rows}"
    )


fact_order: new_rows_found=3
fact_order_detail: new_rows_found=10
fact_payment: new_rows_found=3
fact_shipment: new_rows_found=3


In [34]:
# ============================================================
# STEP 27 - SAVE INCREMENTAL CHECKPOINT
# ============================================================

import boto3

s3_client = boto3.client("s3")

s3_client.put_object(
    Bucket="mahmoud-sic-ecommerce-2026",
    Key="api_raw/checkpoint/last_order_id.txt",
    Body=b"50003",
    ContentType="text/plain"
)

print(
    "Checkpoint saved successfully:"
)
print(
    "s3://mahmoud-sic-ecommerce-2026/"
    "api_raw/checkpoint/last_order_id.txt"
)
print("last_order_id = 50003")


Checkpoint saved successfully:
s3://mahmoud-sic-ecommerce-2026/api_raw/checkpoint/last_order_id.txt
last_order_id = 50003


In [35]:
checkpoint_object = s3_client.get_object(
    Bucket="mahmoud-sic-ecommerce-2026",
    Key="api_raw/checkpoint/last_order_id.txt"
)

checkpoint_value = (
    checkpoint_object["Body"]
    .read()
    .decode("utf-8")
    .strip()
)

print("Checkpoint value:", checkpoint_value)


Checkpoint value: 50003


In [1]:
BUCKET = "mahmoud-sic-ecommerce-2026"

RAW_PATH = f"s3://{BUCKET}/raw/ecommerce"

PROCESSED_PATH = f"s3://{BUCKET}/processed/ecommerce"

REPORT_PATH = f"{PROCESSED_PATH}/_quality_reports"

REJECTED_PATH = f"{PROCESSED_PATH}/_rejected"


Welcome to the Glue Interactive Sessions Kernel
For more information on available magic commands, please type %help in any new cell.

Please view our Getting Started page to access the most up-to-date information on the Interactive Sessions kernel: https://docs.aws.amazon.com/glue/latest/dg/interactive-sessions.html
Installed kernel version: 1.0.10 
Trying to create a Glue session for the kernel.
Session Type: glueetl
Session ID: 6c168906-d72f-4f8f-8061-a3b275ceacb4
Applying the following default arguments:
--glue_kernel_version 1.0.10
--enable-glue-datacatalog true
Waiting for session 6c168906-d72f-4f8f-8061-a3b275ceacb4 to get into ready status...
Session 6c168906-d72f-4f8f-8061-a3b275ceacb4 has been created.



In [2]:
# 2. Read all raw CSV files
customers = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(f"{RAW_PATH}/customers.csv")
)

categories = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(f"{RAW_PATH}/categories.csv")
)

products = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(f"{RAW_PATH}/products.csv")
)

departments = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(f"{RAW_PATH}/departments.csv")
)

employees = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(f"{RAW_PATH}/employees.csv")
)

suppliers = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(f"{RAW_PATH}/suppliers.csv")
)

orders = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(f"{RAW_PATH}/orders.csv")
)

order_details = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(f"{RAW_PATH}/order_details.csv")
)

payments = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(f"{RAW_PATH}/payments.csv")
)

product_suppliers = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(f"{RAW_PATH}/product_suppliers.csv")
)

shippers = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(f"{RAW_PATH}/shippers.csv")
)

shipments = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(f"{RAW_PATH}/shipments.csv")
)

In [3]:
print("Customers:", customers.count())
print("Categories:", categories.count())
print("Products:", products.count())
print("Departments:", departments.count())
print("Employees:", employees.count())
print("Suppliers:", suppliers.count())
print("Orders:", orders.count())
print("Order Details:", order_details.count())
print("Payments:", payments.count())
print("Product Suppliers:", product_suppliers.count())
print("Shippers:", shippers.count())
print("Shipments:", shipments.count())

Customers: 10000
Categories: 20
Products: 1000
Departments: 10
Employees: 200
Suppliers: 100
Orders: 50000
Order Details: 100000
Payments: 45000
Product Suppliers: 2027
Shippers: 10
Shipments: 40000


In [4]:
# Part 1 - Profiling all tables

tables = {
    "customers": customers,
    "categories": categories,
    "products": products,
    "departments": departments,
    "employees": employees,
    "suppliers": suppliers,
    "orders": orders,
    "order_details": order_details,
    "payments": payments,
    "product_suppliers": product_suppliers,
    "shippers": shippers,
    "shipments": shipments
}

for table_name, df in tables.items():
    print("=" * 80)
    print(f"TABLE: {table_name}")
    print("- Schema:")
    df.printSchema()
    print("- First 10 records:")
    df.show(10, truncate=False)


TABLE: customers
- Schema:
root
 |-- CustomerID: integer (nullable = true)
 |-- FirstName: string (nullable = true)
 |-- LastName: string (nullable = true)
 |-- Email: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- RegistrationDate: timestamp (nullable = true)

- First 10 records:
+----------+---------+--------+-----------------------------+----------+--------------+-------------------+
|CustomerID|FirstName|LastName|Email                        |City      |Country       |RegistrationDate   |
+----------+---------+--------+-----------------------------+----------+--------------+-------------------+
|1         |Danielle |Johnson |danielle.johnson1@example.com|Giza      |Egypt         |2023-01-31 00:00:00|
|2         |Joshua   |Walker  |joshua.walker2@example.com   |Dubai     |Jordan        |2021-07-25 00:00:00|
|3         |Jill     |Rhodes  |jill.rhodes3@example.com     |Giza      |United Kingdom|2020-12-22 00:00:00|
|4         |

In [12]:
from pyspark.sql import functions as F

# NULL values in every column of customers
customer_nulls = customers.select([
    F.sum(F.col(column_name).isNull().cast("int")).alias(column_name)
    for column_name in customers.columns
])

print("NULL values in customers:")
customer_nulls.show(truncate=False)

# Duplicate customer IDs
duplicate_customers = (
    customers
    .groupBy("CustomerID")
    .count()
    .filter(F.col("count") > 1)
)

print("Duplicate customerid values:")
duplicate_customers.show(truncate=False)

# Duplicate order IDs
duplicate_orders = (
    orders
    .groupBy("OrderID")
    .count()
    .filter(F.col("count") > 1)
)

print("Duplicate orderid values:")
duplicate_orders.show(truncate=False)


NULL values in customers:
+----------+---------+--------+-----+----+-------+----------------+
|CustomerID|FirstName|LastName|Email|City|Country|RegistrationDate|
+----------+---------+--------+-----+----+-------+----------------+
|0         |0        |0       |0    |0   |0      |0               |
+----------+---------+--------+-----+----+-------+----------------+

Duplicate customerid values:
+----------+-----+
|CustomerID|count|
+----------+-----+
+----------+-----+

Duplicate orderid values:
+-------+-----+
|OrderID|count|
+-------+-----+
+-------+-----+


In [13]:
# Part 2 - Data Cleaning

from pyspark.sql import functions as F
from pyspark.sql.types import StringType

# Put all tables in a dictionary
tables = {
    "customers": customers,
    "categories": categories,
    "products": products,
    "departments": departments,
    "employees": employees,
    "suppliers": suppliers,
    "orders": orders,
    "order_details": order_details,
    "payments": payments,
    "product_suppliers": product_suppliers,
    "shippers": shippers,
    "shipments": shipments
}

# 1. Standardize all column names to lowercase
def standardize_column_names(df):
    for old_name in df.columns:
        new_name = old_name.strip().lower().replace(" ", "_").replace("-", "_")
        df = df.withColumnRenamed(old_name, new_name)
    return df

for table_name in tables:
    tables[table_name] = standardize_column_names(tables[table_name])

# Update the individual DataFrame variables
customers = tables["customers"]
categories = tables["categories"]
products = tables["products"]
departments = tables["departments"]
employees = tables["employees"]
suppliers = tables["suppliers"]
orders = tables["orders"]
order_details = tables["order_details"]
payments = tables["payments"]
product_suppliers = tables["product_suppliers"]
shippers = tables["shippers"]
shipments = tables["shipments"]

# 2. Trim leading/trailing spaces from all string columns
for table_name in tables:
    df = tables[table_name]

    for field in df.schema.fields:
        if isinstance(field.dataType, StringType):
            df = df.withColumn(field.name, F.trim(F.col(field.name)))

    tables[table_name] = df

# Update DataFrame variables again
customers = tables["customers"]
categories = tables["categories"]
products = tables["products"]
departments = tables["departments"]
employees = tables["employees"]
suppliers = tables["suppliers"]
orders = tables["orders"]
order_details = tables["order_details"]
payments = tables["payments"]
product_suppliers = tables["product_suppliers"]
shippers = tables["shippers"]
shipments = tables["shipments"]

# 3. Convert customer emails to lowercase
customers = customers.withColumn(
    "email",
    F.lower(F.col("email"))
)

# 4. Convert order statuses to uppercase
orders = orders.withColumn(
    "status",
    F.upper(F.col("status"))
)

# 5. Find invalid customer emails
invalid_emails = customers.filter(
    ~F.col("email").rlike(
        r"^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$"
    )
)

print("Invalid customer emails:")
invalid_emails.show(10, truncate=False)
print("Invalid email count:", invalid_emails.count())

# 6. Find invalid order statuses
valid_statuses = [
    "PENDING",
    "SHIPPED",
    "DELIVERED",
    "CANCELLED"
]

invalid_statuses = orders.filter(
    ~F.col("status").isin(valid_statuses)
)

print("Invalid order statuses:")
invalid_statuses.groupBy("status").count().show(truncate=False)

# 7. Find products with NULL or non-positive prices
invalid_products = products.filter(
    F.col("price").isNull() | (F.col("price") <= 0)
)

print("Products with NULL or non-positive prices:")
invalid_products.show(10, truncate=False)
print("Invalid product count:", invalid_products.count())

# 8. Remove duplicate records based on primary keys
customers = customers.dropDuplicates(["customerid"])
orders = orders.dropDuplicates(["orderid"])

# 9. Remove records with NULL primary keys
primary_keys = {
    "customers": "customerid",
    "categories": "categoryid",
    "products": "productid",
    "departments": "departmentid",
    "employees": "employeeid",
    "suppliers": "supplierid",
    "orders": "orderid",
    "order_details": "orderdetailid",
    "payments": "paymentid",
    "product_suppliers": "productsupplierid",
    "shippers": "shipperid",
    "shipments": "shipmentid"
}

for table_name, primary_key in primary_keys.items():
    if primary_key in tables[table_name].columns:
        tables[table_name] = tables[table_name].filter(
            F.col(primary_key).isNotNull()
        )

# Update cleaned tables
customers = tables["customers"]
categories = tables["categories"]
products = tables["products"]
departments = tables["departments"]
employees = tables["employees"]
suppliers = tables["suppliers"]
orders = tables["orders"]
order_details = tables["order_details"]
payments = tables["payments"]
product_suppliers = tables["product_suppliers"]
shippers = tables["shippers"]
shipments = tables["shipments"]

print("Data cleaning completed successfully.")


Invalid customer emails:
+----------+---------+--------+-----+----+-------+----------------+
|customerid|firstname|lastname|email|city|country|registrationdate|
+----------+---------+--------+-----+----+-------+----------------+
+----------+---------+--------+-----+----+-------+----------------+

Invalid email count: 0
Invalid order statuses:
+----------+-----+
|status    |count|
+----------+-----+
|PROCESSING|9968 |
+----------+-----+

Products with NULL or non-positive prices:
+---------+----------+-----------+-----+-----+----+-----+
|productid|categoryid|productname|brand|price|cost|stock|
+---------+----------+-----------+-----+-----+----+-----+
+---------+----------+-----------+-----+-----+----+-----+

Invalid product count: 0
Data cleaning completed successfully.


In [14]:
# Part 3 - Data Type Transformation

from pyspark.sql import functions as F

# 1. Convert all ID columns to long
for table_name, df in tables.items():
    for column_name in df.columns:
        if column_name.endswith("id"):
            df = df.withColumn(
                column_name,
                F.col(column_name).cast("long")
            )
    tables[table_name] = df

# 2. Convert price and money columns to decimal(12,2)

products = products \
    .withColumn("price", F.col("price").cast("decimal(12,2)")) \
    .withColumn("cost", F.col("cost").cast("decimal(12,2)"))

order_details = order_details \
    .withColumn("unitprice", F.col("unitprice").cast("decimal(12,2)")) \
    .withColumn("discount", F.col("discount").cast("decimal(12,2)"))

payments = payments.withColumn(
    "amount",
    F.col("amount").cast("decimal(12,2)")
)

# 3. Convert date columns to Spark date
customers = customers.withColumn(
    "registrationdate",
    F.to_date("registrationdate")
)

employees = employees.withColumn(
    "hiredate",
    F.to_date("hiredate")
)

orders = orders.withColumn(
    "orderdate",
    F.to_date("orderdate")
)

payments = payments.withColumn(
    "paymentdate",
    F.to_date("paymentdate")
)

shipments = shipments \
    .withColumn("shipdate", F.to_date("shipdate")) \
    .withColumn("deliverydate", F.to_date("deliverydate"))

# 4. Convert quantity to integer
order_details = order_details.withColumn(
    "quantity",
    F.col("quantity").cast("int")
)

# 5. Extract date parts from orderdate
orders = orders \
    .withColumn("year", F.year("orderdate")) \
    .withColumn("month", F.month("orderdate")) \
    .withColumn("quarter", F.quarter("orderdate")) \
    .withColumn("day", F.dayofmonth("orderdate")) \
    .withColumn("day_of_week", F.dayofweek("orderdate"))

# 6. Recalculate total_amount after converting quantity and unitprice
order_details = order_details.withColumn(
    "total_amount",
    (
        F.col("quantity").cast("decimal(12,2)") *
        F.col("unitprice")
    ).cast("decimal(12,2)")
)

# 7. Update the tables dictionary
tables["customers"] = customers
tables["products"] = products
tables["orders"] = orders
tables["order_details"] = order_details
tables["payments"] = payments
tables["employees"] = employees
tables["shipments"] = shipments

print("Data type transformation completed successfully.")


Data type transformation completed successfully.


In [15]:
print("CUSTOMERS SCHEMA")
customers.printSchema()

print("ORDERS SCHEMA")
orders.printSchema()

print("ORDER_DETAILS SCHEMA")
order_details.printSchema()

print("PAYMENTS SCHEMA")
payments.printSchema()


CUSTOMERS SCHEMA
root
 |-- customerid: integer (nullable = true)
 |-- firstname: string (nullable = true)
 |-- lastname: string (nullable = true)
 |-- email: string (nullable = true)
 |-- city: string (nullable = true)
 |-- country: string (nullable = true)
 |-- registrationdate: date (nullable = true)

ORDERS SCHEMA
root
 |-- orderid: integer (nullable = true)
 |-- customerid: integer (nullable = true)
 |-- orderdate: date (nullable = true)
 |-- status: string (nullable = true)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- quarter: integer (nullable = true)
 |-- day: integer (nullable = true)
 |-- day_of_week: integer (nullable = true)

ORDER_DETAILS SCHEMA
root
 |-- orderdetailid: integer (nullable = true)
 |-- orderid: integer (nullable = true)
 |-- productid: integer (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unitprice: decimal(12,2) (nullable = true)
 |-- discount: decimal(12,2) (nullable = true)
 |-- total_amount: decimal(12,

In [17]:
# Part 4 - Business Transformations

from pyspark.sql import functions as F

# 1. Create full_name in customers
customers = customers.withColumn(
    "full_name",
    F.concat_ws(" ", F.col("firstname"), F.col("lastname"))
)

# 2. Calculate total amount for every order detail
order_details = order_details.withColumn(
    "total_amount",
    (
        F.col("quantity").cast("decimal(12,2)") *
        F.col("unitprice")
    ).cast("decimal(12,2)")
)

# 3. Create the sales fact dataset
fact_sales = (
    orders.alias("o")
    .join(
        order_details.alias("od"),
        F.col("o.orderid") == F.col("od.orderid"),
        "inner"
    )
    .join(
        products.alias("p"),
        F.col("od.productid") == F.col("p.productid"),
        "left"
    )
    .join(
        categories.alias("c"),
        F.col("p.categoryid") == F.col("c.categoryid"),
        "left"
    )
    .join(
        customers.alias("cu"),
        F.col("o.customerid") == F.col("cu.customerid"),
        "left"
    )
    .select(
        F.col("o.orderid").alias("orderid"),
        F.col("o.customerid").alias("customerid"),
        F.col("od.productid").alias("productid"),
        F.col("p.categoryid").alias("categoryid"),
        F.col("o.orderdate").alias("orderdate"),
        F.col("o.year").alias("year"),
        F.col("o.month").alias("month"),
        F.col("o.quarter").alias("quarter"),
        F.col("o.day").alias("day"),
        F.col("o.day_of_week").alias("day_of_week"),
        F.col("od.quantity").alias("quantity"),
        F.col("od.unitprice").alias("unitprice"),
        F.col("od.total_amount").alias("total_amount"),
        F.col("o.status").alias("status"),
        F.col("p.productname").alias("productname"),
        F.col("c.categoryname").alias("categoryname"),
        F.col("cu.full_name").alias("customer_name"),
        F.col("cu.city").alias("city"),
        F.col("cu.country").alias("country")
    )
)

# 4. Calculate total amount and total quantity for every order
order_totals = (
    fact_sales
    .groupBy("orderid")
    .agg(
        F.sum("total_amount")
         .cast("decimal(12,2)")
         .alias("order_total"),
        F.sum("quantity")
         .cast("long")
         .alias("order_quantity")
    )
)

# 5. Add order totals and order value category to orders
orders = (
    orders
    .join(order_totals, on="orderid", how="left")
    .withColumn(
        "order_value_category",
        F.when(F.col("order_total") >= 1000, "HIGH")
         .when(F.col("order_total") >= 500, "MEDIUM")
         .otherwise("LOW")
    )
)

# 6. Calculate total sales for every customer
customer_sales_base = (
    fact_sales
    .groupBy("customerid")
    .agg(
        F.sum("total_amount")
         .cast("decimal(14,2)")
         .alias("total_sales"),
        F.countDistinct("orderid").alias("order_count"),
        F.sum("quantity").cast("long").alias("total_quantity")
    )
)

# 7. Create customer segments
# Assumption documented because the assignment did not define numeric thresholds.
customer_sales = (
    customers
    .select("customerid", "full_name", "city", "country")
    .join(customer_sales_base, on="customerid", how="left")
    .fillna({
        "total_sales": 0,
        "order_count": 0,
        "total_quantity": 0
    })
    .withColumn(
        "customer_segment",
        F.when(F.col("total_sales") >= 10000, "VIP")
         .when(F.col("total_sales") >= 5000, "PREMIUM")
         .when(F.col("total_sales") >= 1000, "REGULAR")
         .otherwise("LOW_VALUE")
    )
)

# 8. Average product price
average_product_price = products.select(
    F.avg("price").cast("decimal(12,2)").alias("average_product_price")
)

print("Business transformations completed successfully.")
print("fact_sales rows:", fact_sales.count())
print("customer_sales rows:", customer_sales.count())

print("Average product price:")
average_product_price.show()


Business transformations completed successfully.
fact_sales rows: 100000
customer_sales rows: 10000
Average product price:
+---------------------+
|average_product_price|
+---------------------+
|              1530.10|
+---------------------+


In [35]:
customers.printSchema()
orders.printSchema()
order_details.printSchema()
products.printSchema()

root
 |-- CustomerID: integer (nullable = true)
 |-- FirstName: string (nullable = true)
 |-- LastName: string (nullable = true)
 |-- Email: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- RegistrationDate: timestamp (nullable = true)

root
 |-- OrderID: integer (nullable = true)
 |-- CustomerID: integer (nullable = true)
 |-- OrderDate: timestamp (nullable = true)
 |-- Status: string (nullable = true)

root
 |-- OrderDetailID: integer (nullable = true)
 |-- OrderID: integer (nullable = true)
 |-- ProductID: integer (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- UnitPrice: double (nullable = true)
 |-- Discount: integer (nullable = true)

root
 |-- ProductID: integer (nullable = true)
 |-- CategoryID: integer (nullable = true)
 |-- ProductName: string (nullable = true)
 |-- Brand: string (nullable = true)
 |-- Price: double (nullable = true)
 |-- Cost: double (nullable = true)
 |-- Stock: integer (nullable = true

In [18]:
# Part 5 - Window Functions

from pyspark.sql import functions as F
from pyspark.sql.window import Window

# 1. Latest order for every customer
latest_order_window = (
    Window
    .partitionBy("customerid")
    .orderBy(
        F.col("orderdate").desc(),
        F.col("orderid").desc()
    )
)

latest_orders = (
    orders
    .withColumn(
        "row_number",
        F.row_number().over(latest_order_window)
    )
    .filter(F.col("row_number") == 1)
    .select(
        "customerid",
        "orderid",
        "status",
        "orderdate"
    )
)

# 2. First order for every customer
first_order_window = (
    Window
    .partitionBy("customerid")
    .orderBy(
        F.col("orderdate").asc(),
        F.col("orderid").asc()
    )
)

first_orders = (
    orders
    .withColumn(
        "row_number",
        F.row_number().over(first_order_window)
    )
    .filter(F.col("row_number") == 1)
    .select(
        "customerid",
        "orderid",
        "status",
        "orderdate"
    )
)

# 3. Rank customers by total sales
customer_rank_window = Window.orderBy(
    F.col("total_sales").desc()
)

ranked_customers = (
    customer_sales
    .withColumn(
        "sales_rank",
        F.rank().over(customer_rank_window)
    )
)

# 4. Top 3 products in every category by sales
product_sales_by_category = (
    fact_sales
    .groupBy(
        "categoryid",
        "categoryname",
        "productid",
        "productname"
    )
    .agg(
        F.sum("total_amount")
         .cast("decimal(14,2)")
         .alias("product_sales")
    )
)

top_products_window = (
    Window
    .partitionBy("categoryid")
    .orderBy(F.col("product_sales").desc())
)

top_3_products_per_category = (
    product_sales_by_category
    .withColumn(
        "category_rank",
        F.row_number().over(top_products_window)
    )
    .filter(F.col("category_rank") <= 3)
)

# 5. Most expensive product in every category
most_expensive_window = (
    Window
    .partitionBy("categoryid")
    .orderBy(
        F.col("price").desc(),
        F.col("productid").asc()
    )
)

most_expensive_products = (
    products
    .join(
        categories,
        on="categoryid",
        how="left"
    )
    .withColumn(
        "price_rank",
        F.row_number().over(most_expensive_window)
    )
    .filter(F.col("price_rank") == 1)
    .select(
        "categoryid",
        "categoryname",
        "productid",
        "productname",
        "price"
    )
)

# 6. Latest shipment for every order
latest_shipment_window = (
    Window
    .partitionBy("orderid")
    .orderBy(
        F.col("shipdate").desc(),
        F.col("shipmentid").desc()
    )
)

latest_shipments = (
    shipments
    .withColumn(
        "row_number",
        F.row_number().over(latest_shipment_window)
    )
    .filter(F.col("row_number") == 1)
    .drop("row_number")
)

print("Window functions completed successfully.")

print("Latest orders per customer:", latest_orders.count())
print("First orders per customer:", first_orders.count())
print("Top 3 products per category:", top_3_products_per_category.count())
print("Most expensive products per category:", most_expensive_products.count())
print("Latest shipments per order:", latest_shipments.count())

print("Top 10 customers by sales:")
ranked_customers.orderBy("sales_rank").show(10, truncate=False)


Window functions completed successfully.
Latest orders per customer: 9931
First orders per customer: 9931
Top 3 products per category: 60
Most expensive products per category: 20
Latest shipments per order: 27459
Top 10 customers by sales:
+----------+----------------+---------+--------------+-----------+-----------+--------------+----------------+----------+
|customerid|full_name       |city     |country       |total_sales|order_count|total_quantity|customer_segment|sales_rank|
+----------+----------------+---------+--------------+-----------+-----------+--------------+----------------+----------+
|6772      |Angela Stewart  |Abu Dhabi|United Kingdom|368984.90  |12         |211           |VIP             |1         |
|245       |Amy Morrison    |Berlin   |Jordan        |338329.93  |9          |192           |VIP             |2         |
|9895      |Benjamin Holder |Jeddah   |France        |330803.47  |7          |198           |VIP             |3         |
|5446      |Lisa Hogan      

In [19]:
# Part 6 - Joins

from pyspark.sql import functions as F

# 1. Orders + Customers
orders_customers = (
    orders.alias("o")
    .join(
        customers.alias("c"),
        F.col("o.customerid") == F.col("c.customerid"),
        "left"
    )
    .select(
        F.col("o.orderid").alias("orderid"),
        F.col("o.customerid").alias("customerid"),
        F.col("c.full_name").alias("customer_name"),
        F.col("c.city").alias("city"),
        F.col("c.country").alias("country"),
        F.col("o.orderdate").alias("orderdate"),
        F.col("o.status").alias("status")
    )
)

# 2. Orders + Order Details + Products
sales_dataset = (
    orders.alias("o")
    .join(
        order_details.alias("od"),
        F.col("o.orderid") == F.col("od.orderid"),
        "inner"
    )
    .join(
        products.alias("p"),
        F.col("od.productid") == F.col("p.productid"),
        "left"
    )
    .select(
        F.col("o.orderid").alias("orderid"),
        F.col("o.customerid").alias("customerid"),
        F.col("o.orderdate").alias("orderdate"),
        F.col("o.status").alias("status"),
        F.col("od.orderdetailid").alias("orderdetailid"),
        F.col("od.productid").alias("productid"),
        F.col("od.quantity").alias("quantity"),
        F.col("od.unitprice").alias("unitprice"),
        F.col("od.total_amount").alias("total_amount"),
        F.col("p.categoryid").alias("categoryid"),
        F.col("p.productname").alias("productname")
    )
)

# 3. Products + Categories
products_categories = (
    products.alias("p")
    .join(
        categories.alias("c"),
        F.col("p.categoryid") == F.col("c.categoryid"),
        "left"
    )
    .select(
        F.col("p.productid").alias("productid"),
        F.col("p.productname").alias("productname"),
        F.col("p.categoryid").alias("categoryid"),
        F.col("c.categoryname").alias("categoryname"),
        F.col("p.brand").alias("brand"),
        F.col("p.price").alias("price"),
        F.col("p.cost").alias("cost"),
        F.col("p.stock").alias("stock")
    )
)

# 4. Products + Product Suppliers + Suppliers
products_suppliers = (
    products.alias("p")
    .join(
        product_suppliers.alias("ps"),
        F.col("p.productid") == F.col("ps.productid"),
        "left"
    )
    .join(
        suppliers.alias("s"),
        F.col("ps.supplierid") == F.col("s.supplierid"),
        "left"
    )
    .select(
        F.col("p.productid").alias("productid"),
        F.col("p.productname").alias("productname"),
        F.col("ps.supplierid").alias("supplierid"),
        F.col("s.suppliername").alias("suppliername"),
        F.col("s.country").alias("supplier_country")
    )
)

# 5. Orders + Payments
orders_payments = (
    orders.alias("o")
    .join(
        payments.alias("pay"),
        F.col("o.orderid") == F.col("pay.orderid"),
        "left"
    )
    .select(
        F.col("o.orderid").alias("orderid"),
        F.col("o.customerid").alias("customerid"),
        F.col("o.orderdate").alias("orderdate"),
        F.col("o.status").alias("status"),
        F.col("pay.paymentid").alias("paymentid"),
        F.col("pay.paymentmethod").alias("payment_method"),
        F.col("pay.paymentdate").alias("payment_date"),
        F.col("pay.amount").alias("payment_amount")
    )
)

# 6. Orders + Shipments + Shippers
orders_shipments_shippers = (
    orders.alias("o")
    .join(
        shipments.alias("sh"),
        F.col("o.orderid") == F.col("sh.orderid"),
        "left"
    )
    .join(
        shippers.alias("sp"),
        F.col("sh.shipperid") == F.col("sp.shipperid"),
        "left"
    )
    .select(
        F.col("o.orderid").alias("orderid"),
        F.col("o.customerid").alias("customerid"),
        F.col("o.orderdate").alias("orderdate"),
        F.col("o.status").alias("status"),
        F.col("sh.shipmentid").alias("shipmentid"),
        F.col("sh.shipdate").alias("ship_date"),
        F.col("sh.deliverydate").alias("delivery_date"),
        F.col("sp.shipperid").alias("shipperid"),
        F.col("sp.companyname").alias("shipper_name")
    )
)

print("Joins completed successfully.")

print("Orders + Customers:", orders_customers.count())
print("Sales dataset:", sales_dataset.count())
print("Products + Categories:", products_categories.count())
print("Products + Suppliers:", products_suppliers.count())
print("Orders + Payments:", orders_payments.count())
print("Orders + Shipments + Shippers:", orders_shipments_shippers.count())


Joins completed successfully.
Orders + Customers: 50000
Sales dataset: 100000
Products + Categories: 1000
Products + Suppliers: 2027
Orders + Payments: 65333
Orders + Shipments + Shippers: 62541


In [22]:
# Part 7 - Aggregations

from pyspark.sql import functions as F

# Overall metrics

# First calculate one total for each order
order_level_sales = (
    fact_sales
    .groupBy("orderid")
    .agg(
        F.sum("total_amount")
         .cast("decimal(14,2)")
         .alias("order_total"),

        F.sum("quantity")
         .cast("long")
         .alias("order_quantity")
    )
)

# Then calculate overall metrics
overall_metrics = (
    fact_sales
    .agg(
        F.sum("total_amount")
         .cast("decimal(14,2)")
         .alias("total_sales"),

        F.countDistinct("orderid")
         .alias("total_orders"),

        F.sum("quantity")
         .cast("long")
         .alias("total_quantity_sold")
    )
    .crossJoin(
        order_level_sales.agg(
            F.avg("order_total")
             .cast("decimal(14,2)")
             .alias("average_order_value")
        )
    )
)

print("Overall metrics:")
overall_metrics.show(truncate=False)

# Sales by customer
customer_sales = (
    fact_sales
    .groupBy("customerid", "customer_name", "city", "country")
    .agg(
        F.sum("total_amount")
         .cast("decimal(14,2)")
         .alias("total_sales"),

        F.countDistinct("orderid")
         .alias("order_count"),

        F.sum("quantity")
         .cast("long")
         .alias("total_quantity")
    )
    .withColumn(
        "customer_segment",
        F.when(F.col("total_sales") >= 10000, "VIP")
         .when(F.col("total_sales") >= 5000, "PREMIUM")
         .when(F.col("total_sales") >= 1000, "REGULAR")
         .otherwise("LOW_VALUE")
    )
)

# Sales by product
product_sales = (
    fact_sales
    .groupBy("productid", "productname")
    .agg(
        F.sum("total_amount")
         .cast("decimal(14,2)")
         .alias("total_sales"),

        F.sum("quantity")
         .cast("long")
         .alias("total_quantity_sold"),

        F.countDistinct("orderid")
         .alias("order_count")
    )
)

# Sales by category
category_sales = (
    fact_sales
    .groupBy("categoryid", "categoryname")
    .agg(
        F.sum("total_amount")
         .cast("decimal(14,2)")
         .alias("total_sales"),

        F.sum("quantity")
         .cast("long")
         .alias("total_quantity_sold"),

        F.countDistinct("orderid")
         .alias("order_count")
    )
)

# Sales by country
country_sales = (
    fact_sales
    .groupBy("country")
    .agg(
        F.sum("total_amount")
         .cast("decimal(14,2)")
         .alias("total_sales"),

        F.countDistinct("orderid")
         .alias("order_count")
    )
)

# Sales by city
city_sales = (
    fact_sales
    .groupBy("city")
    .agg(
        F.sum("total_amount")
         .cast("decimal(14,2)")
         .alias("total_sales"),

        F.countDistinct("orderid")
         .alias("order_count")
    )
)

# Monthly sales
monthly_sales = (
    fact_sales
    .groupBy("year", "month")
    .agg(
        F.sum("total_amount")
         .cast("decimal(14,2)")
         .alias("total_sales"),

        F.countDistinct("orderid")
         .alias("order_count"),

        F.sum("quantity")
         .cast("long")
         .alias("total_quantity_sold")
    )
    .orderBy("year", "month")
)

# Yearly sales
yearly_sales = (
    fact_sales
    .groupBy("year")
    .agg(
        F.sum("total_amount")
         .cast("decimal(14,2)")
         .alias("total_sales"),

        F.countDistinct("orderid")
         .alias("order_count"),

        F.sum("quantity")
         .cast("long")
         .alias("total_quantity_sold")
    )
    .orderBy("year")
)

# Sales by order status
status_sales = (
    fact_sales
    .groupBy("status")
    .agg(
        F.sum("total_amount")
         .cast("decimal(14,2)")
         .alias("total_sales"),

        F.countDistinct("orderid")
         .alias("order_count"),

        F.sum("quantity")
         .cast("long")
         .alias("total_quantity_sold")
    )
    .orderBy("status")
)

# Number of orders per customer
orders_per_customer = (
    orders
    .groupBy("customerid")
    .agg(
        F.countDistinct("orderid").alias("order_count")
    )
)

print("Aggregation datasets created successfully.")

print("customer_sales rows:", customer_sales.count())
print("product_sales rows:", product_sales.count())
print("category_sales rows:", category_sales.count())
print("monthly_sales rows:", monthly_sales.count())

print("Top 5 categories by sales:")
category_sales.orderBy(F.col("total_sales").desc()).show(5, truncate=False)

print("Monthly sales:")
monthly_sales.show(10, truncate=False)


Overall metrics:
+------------+------------+-------------------+-------------------+
|total_sales |total_orders|total_quantity_sold|average_order_value|
+------------+------------+-------------------+-------------------+
|840507631.75|43136       |549473             |19485.06           |
+------------+------------+-------------------+-------------------+

Aggregation datasets created successfully.
customer_sales rows: 9859
product_sales rows: 1000
category_sales rows: 20
monthly_sales rows: 33
Top 5 categories by sales:
+----------+------------+-----------+-------------------+-----------+
|categoryid|categoryname|total_sales|total_quantity_sold|order_count|
+----------+------------+-----------+-------------------+-----------+
|12        |Gaming      |57306656.35|37546              |6437       |
|1         |Electronics |56420102.49|35271              |6020       |
|6         |Clothing    |52495546.95|28916              |4963       |
|19        |Jewelry     |49331847.38|28486            

In [23]:
# Part 8 - Advanced Analytics

from pyspark.sql import functions as F

# 1. Top 10 customers by sales
top_10_customers = (
    customer_sales
    .orderBy(F.col("total_sales").desc())
    .limit(10)
)

# 2. Top 10 products by sales
top_10_products = (
    product_sales
    .orderBy(F.col("total_sales").desc())
    .limit(10)
)

# 3. Top 5 categories by sales
top_5_categories = (
    category_sales
    .orderBy(F.col("total_sales").desc())
    .limit(5)
)

# 4. Customers who have never placed an order
customers_never_ordered = (
    customers
    .select("customerid", "full_name", "city", "country")
    .join(
        orders.select("customerid").distinct(),
        on="customerid",
        how="left_anti"
    )
)

# 5. Products that have never been ordered
products_never_ordered = (
    products
    .select("productid", "productname", "categoryid", "price")
    .join(
        order_details.select("productid").distinct(),
        on="productid",
        how="left_anti"
    )
)

# 6. Customers with more than 10 orders
customers_more_than_10_orders = (
    orders
    .groupBy("customerid")
    .agg(
        F.countDistinct("orderid").alias("order_count")
    )
    .filter(F.col("order_count") > 10)
    .join(
        customers.select("customerid", "full_name"),
        on="customerid",
        how="left"
    )
    .select("customerid", "full_name", "order_count")
    .orderBy(F.col("order_count").desc())
)

# 7. Calculate order totals at order level
order_totals_for_checks = (
    fact_sales
    .groupBy("orderid")
    .agg(
        F.sum("total_amount")
         .cast("decimal(14,2)")
         .alias("calculated_order_total")
    )
)

# 8. Aggregate payments by order before comparing
payment_totals_for_checks = (
    payments
    .groupBy("orderid")
    .agg(
        F.sum("amount")
         .cast("decimal(14,2)")
         .alias("paid_amount")
    )
)

# 9. Orders where payment amount does not match calculated order total
payment_mismatches = (
    order_totals_for_checks
    .join(
        payment_totals_for_checks,
        on="orderid",
        how="left"
    )
    .withColumn(
        "difference",
        F.round(
            F.col("paid_amount") - F.col("calculated_order_total"),
            2
        )
    )
    .filter(
        F.col("paid_amount").isNull() |
        (F.abs(F.col("difference")) > F.lit(0.01))
    )
)

# 10. Orders that do not have a shipment
orders_without_shipment = (
    orders
    .select("orderid", "customerid", "orderdate", "status")
    .join(
        shipments.select("orderid").distinct(),
        on="orderid",
        how="left_anti"
    )
)

# 11. Shipments without a matching order
shipments_without_order = (
    shipments
    .join(
        orders.select("orderid").distinct(),
        on="orderid",
        how="left_anti"
    )
)

print("Advanced analytics completed successfully.")

print("Top 10 customers:")
top_10_customers.show(10, truncate=False)

print("Top 10 products:")
top_10_products.show(10, truncate=False)

print("Top 5 categories:")
top_5_categories.show(5, truncate=False)

print("Customers never ordered:", customers_never_ordered.count())
print("Products never ordered:", products_never_ordered.count())
print("Customers with more than 10 orders:",
      customers_more_than_10_orders.count())
print("Payment mismatches:", payment_mismatches.count())
print("Orders without shipment:", orders_without_shipment.count())
print("Shipments without matching order:",
      shipments_without_order.count())


Advanced analytics completed successfully.
Top 10 customers:
+----------+----------------+---------+--------------+-----------+-----------+--------------+----------------+
|customerid|customer_name   |city     |country       |total_sales|order_count|total_quantity|customer_segment|
+----------+----------------+---------+--------------+-----------+-----------+--------------+----------------+
|6772      |Angela Stewart  |Abu Dhabi|United Kingdom|368984.90  |12         |211           |VIP             |
|245       |Amy Morrison    |Berlin   |Jordan        |338329.93  |9          |192           |VIP             |
|9895      |Benjamin Holder |Jeddah   |France        |330803.47  |7          |198           |VIP             |
|5446      |Lisa Hogan      |Giza     |Jordan        |321600.74  |9          |184           |VIP             |
|5688      |Jeffrey Ramirez |Riyadh   |Egypt         |319630.49  |11         |169           |VIP             |
|4971      |Michael Russell |London   |Jordan      

In [24]:
# Part 9 - Date Analysis

from pyspark.sql import functions as F

# 1. Daily sales
daily_sales = (
    fact_sales
    .groupBy("orderdate")
    .agg(
        F.sum("total_amount")
         .cast("decimal(14,2)")
         .alias("total_sales"),

        F.countDistinct("orderid")
         .alias("order_count"),

        F.sum("quantity")
         .cast("long")
         .alias("total_quantity_sold")
    )
    .orderBy("orderdate")
)

# 2. Monthly sales
monthly_sales_date = (
    fact_sales
    .groupBy(
        F.year("orderdate").alias("year"),
        F.month("orderdate").alias("month")
    )
    .agg(
        F.sum("total_amount")
         .cast("decimal(14,2)")
         .alias("total_sales"),

        F.countDistinct("orderid")
         .alias("order_count"),

        F.sum("quantity")
         .cast("long")
         .alias("total_quantity_sold")
    )
    .orderBy("year", "month")
)

# 3. Quarterly sales
quarterly_sales = (
    fact_sales
    .groupBy(
        F.year("orderdate").alias("year"),
        F.quarter("orderdate").alias("quarter")
    )
    .agg(
        F.sum("total_amount")
         .cast("decimal(14,2)")
         .alias("total_sales"),

        F.countDistinct("orderid")
         .alias("order_count"),

        F.sum("quantity")
         .cast("long")
         .alias("total_quantity_sold")
    )
    .orderBy("year", "quarter")
)

# 4. Yearly sales
yearly_sales_date = (
    fact_sales
    .groupBy(F.year("orderdate").alias("year"))
    .agg(
        F.sum("total_amount")
         .cast("decimal(14,2)")
         .alias("total_sales"),

        F.countDistinct("orderid")
         .alias("order_count"),

        F.sum("quantity")
         .cast("long")
         .alias("total_quantity_sold")
    )
    .orderBy("year")
)

# 5. Month with the highest sales
highest_sales_month = (
    monthly_sales_date
    .orderBy(F.col("total_sales").desc())
    .limit(1)
)

# 6. Day with the highest number of orders
# Use orders, not fact_sales, so order details do not duplicate the count.
daily_order_counts = (
    orders
    .groupBy("orderdate")
    .agg(
        F.countDistinct("orderid").alias("order_count")
    )
)

highest_order_day = (
    daily_order_counts
    .orderBy(
        F.col("order_count").desc(),
        F.col("orderdate").asc()
    )
    .limit(1)
)

# 7. Average number of orders per month
orders_by_month = (
    orders
    .groupBy(
        F.year("orderdate").alias("year"),
        F.month("orderdate").alias("month")
    )
    .agg(
        F.countDistinct("orderid").alias("order_count")
    )
)

average_orders_per_month = (
    orders_by_month
    .agg(
        F.avg("order_count")
         .cast("decimal(14,2)")
         .alias("average_orders_per_month")
    )
)

print("Date analysis completed successfully.")

print("Daily sales rows:", daily_sales.count())
print("Monthly sales rows:", monthly_sales_date.count())
print("Quarterly sales rows:", quarterly_sales.count())
print("Yearly sales rows:", yearly_sales_date.count())

print("Month with highest sales:")
highest_sales_month.show(truncate=False)

print("Day with highest number of orders:")
highest_order_day.show(truncate=False)

print("Average orders per month:")
average_orders_per_month.show(truncate=False)


Date analysis completed successfully.
Daily sales rows: 989
Monthly sales rows: 33
Quarterly sales rows: 11
Yearly sales rows: 3
Month with highest sales:
+----+-----+-----------+-----------+-------------------+
|year|month|total_sales|order_count|total_quantity_sold|
+----+-----+-----------+-----------+-------------------+
|2025|3    |27889327.64|1389       |18069              |
+----+-----+-----------+-----------+-------------------+

Day with highest number of orders:
+----------+-----------+
|orderdate |order_count|
+----------+-----------+
|2024-09-10|75         |
+----------+-----------+

Average orders per month:
+------------------------+
|average_orders_per_month|
+------------------------+
|1515.15                 |
+------------------------+


In [25]:
# Part 10 - Write final outputs as Parquet with Snappy

OUTPUT_PATH = f"s3://{BUCKET}/processed/ecommerce"

def write_parquet(df, path, partition_cols=None):
    writer = (
        df.write
        .mode("overwrite")
        .format("parquet")
        .option("compression", "snappy")
    )

    if partition_cols:
        writer = writer.partitionBy(*partition_cols)

    writer.save(path)

# 1. Cleaned base datasets
write_parquet(customers, f"{OUTPUT_PATH}/customers")
write_parquet(products, f"{OUTPUT_PATH}/products")
write_parquet(categories, f"{OUTPUT_PATH}/categories")

# Orders must be partitioned by year and month
write_parquet(
    orders,
    f"{OUTPUT_PATH}/orders",
    partition_cols=["year", "month"]
)

write_parquet(order_details, f"{OUTPUT_PATH}/order_details")
write_parquet(payments, f"{OUTPUT_PATH}/payments")
write_parquet(shipments, f"{OUTPUT_PATH}/shipments")

# 2. Analytical datasets
write_parquet(fact_sales, f"{OUTPUT_PATH}/fact_sales")
write_parquet(customer_sales, f"{OUTPUT_PATH}/customer_sales")
write_parquet(product_sales, f"{OUTPUT_PATH}/product_sales")
write_parquet(category_sales, f"{OUTPUT_PATH}/category_sales")
write_parquet(monthly_sales, f"{OUTPUT_PATH}/monthly_sales")

print("All required datasets were written successfully.")
print(f"Output path: {OUTPUT_PATH}")


All required datasets were written successfully.
Output path: s3://mahmoud-sic-ecommerce-2026/processed/ecommerce


In [36]:
customers.show(10, truncate=False)
orders.show(10, truncate=False)

+----------+---------+---------+-----------------------------+----------+-------------+-------------------+
|CustomerID|FirstName|LastName |Email                        |City      |Country      |RegistrationDate   |
+----------+---------+---------+-----------------------------+----------+-------------+-------------------+
|1         |Kristen  |Roberts  |kristen.roberts1@example.com |Abu Dhabi |United States|2026-09-13 00:00:00|
|2         |Sara     |Gill     |sara.gill2@example.com       |Berlin    |United States|2026-01-10 00:00:00|
|3         |Cody     |Burns    |cody.burns3@example.com      |Amman     |France       |2026-02-09 00:00:00|
|4         |John     |Anderson |john.anderson4@example.com   |Abu Dhabi |Jordan       |2021-11-14 00:00:00|
|5         |Jeanette |Fuller   |jeanette.fuller5@example.com |Giza      |United States|2026-09-07 00:00:00|
|6         |Dylan    |Fernandez|dylan.fernandez6@example.com |Berlin    |Saudi Arabia |2020-11-02 00:00:00|
|7         |Lori     |Griffi

In [37]:
def clean_column_names(df):
    for column in df.columns:
        new_name = (
            column.strip()
            .lower()
            .replace(" ", "_")
            .replace("-", "_")
        )
        
        df = df.withColumnRenamed(column, new_name)
    
    return df

In [38]:
customers = clean_column_names(customers)
categories = clean_column_names(categories)
products = clean_column_names(products)
departments = clean_column_names(departments)
employees = clean_column_names(employees)
suppliers = clean_column_names(suppliers)
orders = clean_column_names(orders)
order_details = clean_column_names(order_details)
payments = clean_column_names(payments)
product_suppliers = clean_column_names(product_suppliers)
shippers = clean_column_names(shippers)
shipments = clean_column_names(shipments)

In [39]:
customers.printSchema()

root
 |-- customerid: integer (nullable = true)
 |-- firstname: string (nullable = true)
 |-- lastname: string (nullable = true)
 |-- email: string (nullable = true)
 |-- city: string (nullable = true)
 |-- country: string (nullable = true)
 |-- registrationdate: timestamp (nullable = true)


In [40]:
departments.printSchema()

root
 |-- departmentid: integer (nullable = true)
 |-- departmentname: string (nullable = true)


In [41]:
customers = customers.dropDuplicates(["customerid"])

orders = orders.dropDuplicates(["orderid"])

products = products.dropDuplicates(["productid"])

order_details = order_details.dropDuplicates(["orderdetailid"])

In [14]:
customers.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in customers.columns
]).show()

+----------+---------+--------+-----+----+-------+----------------+
|customerid|firstname|lastname|email|city|country|registrationdate|
+----------+---------+--------+-----+----+-------+----------------+
|         0|        0|       0|    0|   0|      0|               0|
+----------+---------+--------+-----+----+-------+----------------+


In [42]:
customers = (
    customers
    .withColumn("firstname", F.trim("firstname"))
    .withColumn("lastname", F.trim("lastname"))
    .withColumn("email", F.lower(F.trim("email")))
    .withColumn("city", F.trim("city"))
    .withColumn("country", F.trim("country"))
)

In [43]:
orders = (
    orders
    .withColumn("status", F.upper(F.trim("status")))
)

In [45]:
orders = (
    orders
    .withColumn(
        "orderid",
        F.col("orderid").cast("long")
    )
    .withColumn(
        "customerid",
        F.col("customerid").cast("long")
    )
    .withColumn(
        "orderdate",
        F.to_date("orderdate")
    )
)

In [46]:
customers = customers.withColumn(
    "valid_email",
    F.when(
        F.col("email").rlike(
            r"^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$"
        ),
        True
    ).otherwise(False)
)

In [47]:
orders = orders.withColumn(
    "status",
    F.when(
        F.col("status").isin(
            "PENDING",
            "SHIPPED",
            "DELIVERED",
            "CANCELLED"
        ),
        F.col("status")
    ).otherwise("UNKNOWN")
)

In [48]:
order_details = order_details.withColumn(
    "total_amount",
    F.col("quantity") * F.col("unitprice")
)

In [49]:
order_details = order_details.withColumn(
    "quantity",
    F.when(
        F.col("quantity") < 0,
        0
    ).otherwise(F.col("quantity"))
)

In [50]:
order_details = order_details.withColumn(
    "unitprice",
    F.when(
        F.col("unitprice") < 0,
        0
    ).otherwise(F.col("unitprice"))
)

In [51]:
orders = (
    orders
    .withColumn("year", F.year("orderdate"))
    .withColumn("month", F.month("orderdate"))
    .withColumn("day", F.dayofmonth("orderdate"))
    .withColumn("quarter", F.quarter("orderdate"))
    .withColumn("day_of_week", F.dayofweek("orderdate"))
)

In [52]:
order_sales = (
    orders.alias("o")
    .join(
        order_details.alias("od"),
        F.col("o.orderid") == F.col("od.orderid"),
        "inner"
    )
)

In [53]:
order_sales = order_sales.select(
    F.col("o.orderid"),
    F.col("o.customerid"),
    F.col("o.orderdate"),
    F.col("o.status"),
    F.col("od.productid"),
    F.col("od.quantity"),
    F.col("od.unitprice"),
    F.col("od.total_amount")
)

In [27]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Define the window
customer_window = (
    Window
    .partitionBy("customerid")
    .orderBy(F.col("orderdate").desc())
)

# Add row number
last_order_customer = (
    orders
    .withColumn(
        "last_order_rank",
        F.row_number().over(customer_window)
    )
)

# Keep only the latest order for each customer
last_order_customer = (
    last_order_customer
    .filter(F.col("last_order_rank") == 1)
)

last_order_customer.show()

+-------+----------+----------+---------+----+-----+---+-------+-----------+---------------+
|orderid|customerid| orderdate|   status|year|month|day|quarter|day_of_week|last_order_rank|
+-------+----------+----------+---------+----+-----+---+-------+-----------+---------------+
| 960221|        28|2024-11-06|  PENDING|2024|   11|  6|      4|          4|              1|
| 686734|        29|2026-06-29|  SHIPPED|2026|    6| 29|      2|          2|              1|
| 153064|        30|2026-07-11|DELIVERED|2026|    7| 11|      3|          7|              1|
|1984435|        33|2025-11-26|DELIVERED|2025|   11| 26|      4|          4|              1|
|1324387|        93|2026-06-15|  SHIPPED|2026|    6| 15|      2|          2|              1|
| 822306|       151|2026-04-23|  SHIPPED|2026|    4| 23|      2|          5|              1|
|1301514|       171|2025-09-21|  UNKNOWN|2025|    9| 21|      3|          1|              1|
| 903362|       200|2026-08-09|  UNKNOWN|2026|    8|  9|      3|      

In [54]:
spark.stop()